In [3]:
import polars as pl
import numpy as np
import os
import pandas as pd

In [5]:
def partition_device(ip: str, train_ratio: float = 0.7, data_frac: float = 1):
    safe_ip = ip.replace(":", "_").replace(".", "_")
    print(safe_ip)
    path = f"C:\\Users\\babai\\OneDrive\\Desktop\\CaseStudiesDatasets\\per_device\\device_{safe_ip}.parquet"
    
    lf_full = pl.scan_parquet(path)

    # total rows
    total = lf_full.select(pl.len()).collect().item()
    
    # take only first X% of data (sequential)
    subset_size = int(total * data_frac)
    lf = lf_full.limit(subset_size)

    # now split this subset
    cutoff = int(subset_size * train_ratio)
    
    train_df = (
        lf.with_row_index("_idx")
        .filter(pl.col("_idx") < cutoff)
        .filter(pl.col("detailed-label") == "-")   # benign only
        .drop("_idx")
        .collect()
    )
    
    test_df = (
        lf.with_row_index("_idx")
        .filter(pl.col("_idx") >= cutoff)
        .drop("_idx")
        .collect()
    )
    
    print(f"{ip} — Using {data_frac*100:.1f}% of data ({subset_size:,} rows)")
    print(f"{ip} — Train (benign): {len(train_df):,} rows")
    print(f"{ip} — Test (mixed):   {len(test_df):,} rows")
    
    return train_df, test_df

def df_size_info(df, name="DF"):
    rows = df.height
    cols = df.width
    size_bytes = df.estimated_size()
    size_mb = size_bytes / (1024 ** 2)

    print(f"{name}:")
    print(f"  Rows: {rows:,}")
    print(f"  Columns: {cols}")
    print(f"  Estimated size: {size_mb:.2f} MB\n")

def print_label_distribution(df, col="detailed-label", name="DF"):
    vc = df[col].value_counts().sort("count", descending=True)
    
    total = df.height
    vc = vc.with_columns(
        (pl.col("count") / total * 100).alias("percentage")
    )
    
    print(f"{name} Label Distribution:")
    print(vc)
    print()

In [6]:
train_df,test_df = partition_device("192_168_1_196") 
df_size_info(train_df)
df_size_info(test_df) 
print_label_distribution(train_df)
print_label_distribution(test_df)

192_168_1_196
192_168_1_196 — Using 100.0% of data (10,446,132 rows)
192_168_1_196 — Train (benign): 6,403,650 rows
192_168_1_196 — Test (mixed):   3,133,840 rows
DF:
  Rows: 6,403,650
  Columns: 14
  Estimated size: 403.06 MB

DF:
  Rows: 3,133,840
  Columns: 14
  Estimated size: 197.25 MB

DF Label Distribution:
shape: (1, 3)
┌────────────────┬─────────┬────────────┐
│ detailed-label ┆ count   ┆ percentage │
│ ---            ┆ ---     ┆ ---        │
│ cat            ┆ u32     ┆ f64        │
╞════════════════╪═════════╪════════════╡
│ -              ┆ 6403650 ┆ 100.0      │
└────────────────┴─────────┴────────────┘

DF Label Distribution:
shape: (2, 3)
┌────────────────┬─────────┬────────────┐
│ detailed-label ┆ count   ┆ percentage │
│ ---            ┆ ---     ┆ ---        │
│ cat            ┆ u32     ┆ f64        │
╞════════════════╪═════════╪════════════╡
│ -              ┆ 1857084 ┆ 59.259056  │
│ DDoS           ┆ 1276756 ┆ 40.740944  │
└────────────────┴─────────┴────────────┘



In [4]:
import numpy
import scipy
import sklearn

print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)

NumPy: 1.26.4
SciPy: 1.13.1
Scikit-learn: 1.5.2


In [5]:
print(train_df.schema)
print(train_df.columns)

Schema({'ts': Float64, 'id.orig_h': String, 'id.orig_p': UInt32, 'id.resp_p': UInt32, 'proto': Categorical, 'duration': Float32, 'orig_bytes': Float32, 'resp_bytes': Float32, 'conn_state': Categorical, 'history': String, 'orig_pkts': Float32, 'resp_pkts': Float32, 'label': Categorical, 'detailed-label': Categorical})
['ts', 'id.orig_h', 'id.orig_p', 'id.resp_p', 'proto', 'duration', 'orig_bytes', 'resp_bytes', 'conn_state', 'history', 'orig_pkts', 'resp_pkts', 'label', 'detailed-label']


In [7]:
# ============================================================
# PHASE A — DATA PREPARATION + ENCODING
# ============================================================

import numpy as np
import polars as pl

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


# ============================================================
# 1. CONFIGURATION
# ============================================================

RANDOM_STATE = 42

# ------------------------------------------------------------
# Original IoT-23 features
# ------------------------------------------------------------

ALL_FEATURES = [
    "id.orig_p",
    "id.resp_p",
    "duration",
    "orig_bytes",
    "resp_bytes",
    "orig_pkts",
    "resp_pkts",
    "proto",
    "conn_state",
    "history"
]


NUMERIC_FEATURES = [
    "id.orig_p",
    "id.resp_p",
    "duration",
    "orig_bytes",
    "resp_bytes",
    "orig_pkts",
    "resp_pkts"
]


CATEGORICAL_FEATURES = [
    "proto",
    "conn_state",
    "history"
]


# ============================================================
# 2. EXPERIMENT DATA SIZES
# ============================================================

TRAIN_SAMPLE_SIZE = 10_000

VALIDATION_BENIGN = 2_500
VALIDATION_ATTACK = 2_500

FINAL_BENIGN = 50_000
FINAL_ATTACK = 50_000


# ============================================================
# 3. CHECK INPUT DATA
# ============================================================

print("=" * 70)
print("PHASE A — DATA PREPARATION")
print("=" * 70)

print("\nOriginal features:")

for feature in ALL_FEATURES:
    print("  ", feature)


print("\nTotal original features:", len(ALL_FEATURES))


# ------------------------------------------------------------
# Make sure required columns exist
# ------------------------------------------------------------

required_columns = (
    ALL_FEATURES +
    ["detailed-label"]
)

missing_train = [
    c for c in required_columns
    if c not in train_df.columns
]

missing_test = [
    c for c in required_columns
    if c not in test_df.columns
]

if missing_train:
    raise ValueError(
        f"Missing columns in train_df: {missing_train}"
    )

if missing_test:
    raise ValueError(
        f"Missing columns in test_df: {missing_test}"
    )


# ============================================================
# 4. SEPARATE TEST DATA INTO BENIGN / ATTACK
# ============================================================

benign_test = test_df.filter(
    pl.col("detailed-label") == "-"
)

attack_test = test_df.filter(
    pl.col("detailed-label") != "-"
)


print("\nAvailable test data:")

print(
    f"  Benign : {benign_test.height:,}"
)

print(
    f"  Attack : {attack_test.height:,}"
)


# ============================================================
# 5. CREATE TRAINING SAMPLE
# ============================================================

if train_df.height < TRAIN_SAMPLE_SIZE:

    raise ValueError(
        f"train_df only contains "
        f"{train_df.height:,} rows, but "
        f"{TRAIN_SAMPLE_SIZE:,} requested."
    )


train_sample = train_df.sample(
    n=TRAIN_SAMPLE_SIZE,
    seed=RANDOM_STATE
)


# ============================================================
# 6. CREATE VALIDATION SET
# ============================================================

if benign_test.height < (
    VALIDATION_BENIGN +
    FINAL_BENIGN
):

    raise ValueError(
        "Not enough benign test samples."
    )


if attack_test.height < (
    VALIDATION_ATTACK +
    FINAL_ATTACK
):

    raise ValueError(
        "Not enough attack test samples."
    )


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

validation_benign = benign_test.head(
    VALIDATION_BENIGN
)

validation_attack = attack_test.head(
    VALIDATION_ATTACK
)


validation_df = pl.concat([
    validation_benign,
    validation_attack
])


# ============================================================
# 7. CREATE FINAL TEST SET
# ============================================================

# IMPORTANT:
#
# Final test starts AFTER the validation samples.
#
# Therefore validation and final test do not overlap.

final_benign = benign_test.slice(
    VALIDATION_BENIGN,
    FINAL_BENIGN
)

final_attack = attack_test.slice(
    VALIDATION_ATTACK,
    FINAL_ATTACK
)


final_test_df = pl.concat([
    final_benign,
    final_attack
])


# ============================================================
# 8. SHUFFLE VALIDATION / FINAL TEST
# ============================================================

validation_df = validation_df.sample(
    fraction=1.0,
    seed=RANDOM_STATE
)

final_test_df = final_test_df.sample(
    fraction=1.0,
    seed=RANDOM_STATE
)


# ============================================================
# 9. DATASET INFORMATION
# ============================================================

print("\n")
print("=" * 70)
print("DATASET SPLIT")
print("=" * 70)

print(
    f"\nTraining sample:"
)

print(
    f"  Rows: {train_sample.height:,}"
)

print(
    f"  Labels: BENIGN ONLY"
)


print(
    f"\nValidation set:"
)

print(
    f"  Rows: {validation_df.height:,}"
)

print(
    f"  Benign: {VALIDATION_BENIGN:,}"
)

print(
    f"  Attack: {VALIDATION_ATTACK:,}"
)


print(
    f"\nFinal test set:"
)

print(
    f"  Rows: {final_test_df.height:,}"
)

print(
    f"  Benign: {FINAL_BENIGN:,}"
)

print(
    f"  Attack: {FINAL_ATTACK:,}"
)


# ============================================================
# 10. VALIDATION LABELS
# ============================================================

y_validation = (
    validation_df
    .select(
        pl.when(
            pl.col("detailed-label") == "-"
        )
        .then(0)
        .otherwise(1)
        .alias("label")
    )
    .to_numpy()
    .ravel()
    .astype(np.int8)
)


# ============================================================
# 11. FINAL TEST LABELS
# ============================================================

y_final = (
    final_test_df
    .select(
        pl.when(
            pl.col("detailed-label") == "-"
        )
        .then(0)
        .otherwise(1)
        .alias("label")
    )
    .to_numpy()
    .ravel()
    .astype(np.int8)
)


print("\nValidation labels:")

print(
    f"  Benign: {(y_validation == 0).sum():,}"
)

print(
    f"  Attack: {(y_validation == 1).sum():,}"
)


print("\nFinal test labels:")

print(
    f"  Benign: {(y_final == 0).sum():,}"
)

print(
    f"  Attack: {(y_final == 1).sum():,}"
)


# ============================================================
# 12. CONVERT FEATURES TO PANDAS
# ============================================================

train_pd = train_sample.select(
    ALL_FEATURES
).to_pandas()


validation_pd = validation_df.select(
    ALL_FEATURES
).to_pandas()


final_test_pd = final_test_df.select(
    ALL_FEATURES
).to_pandas()


# ============================================================
# 13. ONE-HOT ENCODER
# ============================================================

print("\n")
print("=" * 70)
print("ONE-HOT ENCODING")
print("=" * 70)

print("\nFitting encoder on training data only...")


encoder = ColumnTransformer(

    transformers=[

        # ----------------------------------------------------
        # Numerical features
        # ----------------------------------------------------

        (
            "num",

            "passthrough",

            NUMERIC_FEATURES
        ),

        # ----------------------------------------------------
        # Categorical features
        # ----------------------------------------------------

        (
            "cat",

            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),

            CATEGORICAL_FEATURES
        )
    ],

    remainder="drop"
)


# ============================================================
# 14. FIT ENCODER
# ============================================================

encoder.fit(
    train_pd
)


print(
    "Encoder fitted."
)


# ============================================================
# 15. TRANSFORM DATA
# ============================================================

X_train = encoder.transform(
    train_pd
)

X_validation = encoder.transform(
    validation_pd
)

X_final = encoder.transform(
    final_test_pd
)


print("\nEncoded shapes:")

print(
    f"  Training    : {X_train.shape}"
)

print(
    f"  Validation  : {X_validation.shape}"
)

print(
    f"  Final test  : {X_final.shape}"
)


# ============================================================
# 16. GET ENCODED FEATURE NAMES
# ============================================================

encoded_feature_names = (
    encoder.get_feature_names_out()
)


print(
    "\nTotal encoded columns:",
    len(encoded_feature_names)
)


# ============================================================
# 17. BUILD ORIGINAL → ENCODED FEATURE MAPPING
# ============================================================

feature_groups = {}


# ------------------------------------------------------------
# Numerical features
# ------------------------------------------------------------

for feature in NUMERIC_FEATURES:

    indices = []

    for i, name in enumerate(
        encoded_feature_names
    ):

        if name == f"num__{feature}":

            indices.append(i)

    feature_groups[feature] = indices


# ------------------------------------------------------------
# Categorical features
# ------------------------------------------------------------

for feature in CATEGORICAL_FEATURES:

    indices = []

    prefix = f"cat__{feature}_"

    for i, name in enumerate(
        encoded_feature_names
    ):

        if name.startswith(prefix):

            indices.append(i)

    feature_groups[feature] = indices


# ============================================================
# 18. PRINT FEATURE MAPPING
# ============================================================

print("\n")
print("=" * 70)
print("FEATURE MAPPING")
print("=" * 70)

for feature in ALL_FEATURES:

    indices = feature_groups[feature]

    print(
        f"\n{feature}"
    )

    print(
        f"  Encoded columns: {len(indices)}"
    )

    for index in indices:

        print(
            f"    [{index}] "
            f"{encoded_feature_names[index]}"
        )


# ============================================================
# 19. FUNCTION FOR ORIGINAL FEATURE SELECTION
# ============================================================

def get_encoded_indices(chromosome):

    """
    Convert a binary chromosome operating on
    the 10 original features into encoded-column
    indices.

    Example:

        chromosome =
        [1,0,1,0,0,0,0,0,1,0]

    selects:

        id.orig_p
        duration
        conn_state

    and returns the corresponding encoded
    column indices.
    """

    selected_indices = []

    for bit, feature in zip(
        chromosome,
        ALL_FEATURES
    ):

        if bit == 1:

            selected_indices.extend(
                feature_groups[feature]
            )

    return sorted(
        selected_indices
    )


# ============================================================
# 20. FUNCTION TO GET ORIGINAL FEATURE NAMES
# ============================================================

def get_selected_features(chromosome):

    return [
        feature
        for feature, bit
        in zip(
            ALL_FEATURES,
            chromosome
        )
        if bit == 1
    ]


# ============================================================
# 21. TEST THE FEATURE MAPPING
# ============================================================

test_chromosome = np.array(
    [1, 0, 1, 0, 0, 0, 0, 0, 1, 0],
    dtype=np.int8
)


test_features = get_selected_features(
    test_chromosome
)

test_indices = get_encoded_indices(
    test_chromosome
)


print("\n")
print("=" * 70)
print("FEATURE MAPPING TEST")
print("=" * 70)

print(
    "\nTest chromosome:"
)

print(
    test_chromosome
)

print(
    "\nSelected original features:"
)

for feature in test_features:

    print(
        "  +",
        feature
    )

print(
    "\nSelected encoded indices:"
)

print(
    test_indices
)


# ============================================================
# 22. VERIFY SELECTED MATRIX
# ============================================================

X_test_selected = X_train[
    :,
    test_indices
]


print(
    "\nSelected training shape:"
)

print(
    X_test_selected.shape
)


# ============================================================
# 23. FINAL PHASE A SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("PHASE A COMPLETE")
print("=" * 70)

print(
    "\nOriginal features:",
    len(ALL_FEATURES)
)

print(
    "Encoded features:",
    X_train.shape[1]
)

print(
    "Training rows:",
    X_train.shape[0]
)

print(
    "Validation rows:",
    X_validation.shape[0]
)

print(
    "Final test rows:",
    X_final.shape[0]
)

print(
    "\nReady for:"
)

print(
    "  1. Full-feature SVM baseline"
)

print(
    "  2. Full-feature KNN baseline"
)

print(
    "  3. Binary GA"
)

print(
    "  4. (1+1)-Evolutionary Strategy"
)

print(
    "  5. Binary Chaotic GA"
)

PHASE A — DATA PREPARATION

Original features:
   id.orig_p
   id.resp_p
   duration
   orig_bytes
   resp_bytes
   orig_pkts
   resp_pkts
   proto
   conn_state
   history

Total original features: 10

Available test data:
  Benign : 1,857,084
  Attack : 1,276,756


DATASET SPLIT

Training sample:
  Rows: 10,000
  Labels: BENIGN ONLY

Validation set:
  Rows: 5,000
  Benign: 2,500
  Attack: 2,500

Final test set:
  Rows: 100,000
  Benign: 50,000
  Attack: 50,000

Validation labels:
  Benign: 2,500
  Attack: 2,500

Final test labels:
  Benign: 50,000
  Attack: 50,000


ONE-HOT ENCODING

Fitting encoder on training data only...
Encoder fitted.

Encoded shapes:
  Training    : (10000, 16)
  Validation  : (5000, 16)
  Final test  : (100000, 16)

Total encoded columns: 16


FEATURE MAPPING

id.orig_p
  Encoded columns: 1
    [0] num__id.orig_p

id.resp_p
  Encoded columns: 1
    [1] num__id.resp_p

duration
  Encoded columns: 1
    [2] num__duration

orig_bytes
  Encoded columns: 1
    [3] 

In [10]:
# ============================================================
# PHASE B — FULL-FEATURE BASELINES
# ============================================================
#
# Baselines:
#   1. One-Class SVM
#   2. KNN anomaly detector
#
# NO FEATURE SELECTION IS USED.
#
# Both models:
#   - Train using 10,000 BENIGN samples
#   - Validation: 2,500 benign + 2,500 attack
#   - Final test: 50,000 benign + 50,000 attack
#
# IMPORTANT:
#   - Scaler fitted ONLY on training data
#   - Threshold determined ONLY from benign validation data
#   - Final test is not used for threshold selection
# ============================================================

import time
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM
from sklearn.neighbors import NearestNeighbors

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

SVM_NU = 0.05
SVM_KERNEL = "linear"

KNN_K = 5

THRESHOLD_PERCENTILE = 99


# ============================================================
# 2. PHASE B HEADER
# ============================================================

print("=" * 70)
print("PHASE B — FULL-FEATURE BASELINES")
print("=" * 70)

print(
    "\nOriginal features:",
    len(ALL_FEATURES)
)

print(
    "Encoded features:",
    X_train.shape[1]
)


# ============================================================
# 3. SCALE FEATURES
# ============================================================
#
# The scaler is fitted ONLY on benign training data.
#
# with_mean=False is used because X_train is sparse.
# ============================================================

print("\nScaling features...")

scaler = StandardScaler(
    with_mean=False
)


scale_start = time.time()


X_train_scaled = scaler.fit_transform(
    X_train
)

X_validation_scaled = scaler.transform(
    X_validation
)

X_final_scaled = scaler.transform(
    X_final
)


scale_time = time.time() - scale_start


print(
    f"Scaling time: {scale_time:.4f} seconds"
)

print(
    "Training shape   :",
    X_train_scaled.shape
)

print(
    "Validation shape :",
    X_validation_scaled.shape
)

print(
    "Final test shape :",
    X_final_scaled.shape
)


# ============================================================
# 4. COMMON EVALUATION FUNCTION
# ============================================================

def evaluate_anomaly_detector(
    model_name,
    validation_scores,
    final_scores,
    y_validation,
    y_final,
    threshold
):

    # --------------------------------------------------------
    # Validation predictions
    # --------------------------------------------------------

    validation_predictions = (
        validation_scores > threshold
    ).astype(np.int8)


    # --------------------------------------------------------
    # Final predictions
    # --------------------------------------------------------

    final_predictions = (
        final_scores > threshold
    ).astype(np.int8)


    # ========================================================
    # VALIDATION METRICS
    # ========================================================

    val_accuracy = accuracy_score(
        y_validation,
        validation_predictions
    )

    val_precision = precision_score(
        y_validation,
        validation_predictions,
        zero_division=0
    )

    val_recall = recall_score(
        y_validation,
        validation_predictions,
        zero_division=0
    )

    val_f1 = f1_score(
        y_validation,
        validation_predictions,
        zero_division=0
    )

    val_auc = roc_auc_score(
        y_validation,
        validation_scores
    )

    val_cm = confusion_matrix(
        y_validation,
        validation_predictions
    )


    # ========================================================
    # FINAL TEST METRICS
    # ========================================================

    final_accuracy = accuracy_score(
        y_final,
        final_predictions
    )

    final_precision = precision_score(
        y_final,
        final_predictions,
        zero_division=0
    )

    final_recall = recall_score(
        y_final,
        final_predictions,
        zero_division=0
    )

    final_f1 = f1_score(
        y_final,
        final_predictions,
        zero_division=0
    )

    final_auc = roc_auc_score(
        y_final,
        final_scores
    )

    final_cm = confusion_matrix(
        y_final,
        final_predictions
    )


    # ========================================================
    # PRINT VALIDATION
    # ========================================================

    print("\n")
    print("=" * 70)
    print(f"{model_name} — VALIDATION")
    print("=" * 70)

    print(
        f"Threshold : {threshold:.10f}"
    )

    print(
        f"Accuracy  : {val_accuracy:.4f}"
    )

    print(
        f"Precision : {val_precision:.4f}"
    )

    print(
        f"Recall    : {val_recall:.4f}"
    )

    print(
        f"F1        : {val_f1:.4f}"
    )

    print(
        f"ROC-AUC   : {val_auc:.4f}"
    )

    print(
        "\nConfusion matrix:"
    )

    print(
        val_cm
    )


    # ========================================================
    # PRINT FINAL TEST
    # ========================================================

    print("\n")
    print("=" * 70)
    print(f"{model_name} — FINAL TEST")
    print("=" * 70)

    print(
        f"Accuracy  : {final_accuracy:.4f}"
    )

    print(
        f"Precision : {final_precision:.4f}"
    )

    print(
        f"Recall    : {final_recall:.4f}"
    )

    print(
        f"F1        : {final_f1:.4f}"
    )

    print(
        f"ROC-AUC   : {final_auc:.4f}"
    )

    print(
        "\nConfusion matrix:"
    )

    print(
        final_cm
    )


    return {

        "model": model_name,

        "features":
            len(ALL_FEATURES),

        "encoded_features":
            X_train.shape[1],

        "threshold":
            threshold,

        "val_accuracy":
            val_accuracy,

        "val_precision":
            val_precision,

        "val_recall":
            val_recall,

        "val_f1":
            val_f1,

        "val_auc":
            val_auc,

        "final_accuracy":
            final_accuracy,

        "final_precision":
            final_precision,

        "final_recall":
            final_recall,

        "final_f1":
            final_f1,

        "final_auc":
            final_auc,

        "validation_cm":
            val_cm,

        "final_cm":
            final_cm
    }


# ============================================================
# 5. BASELINE 1 — ONE-CLASS SVM
# ============================================================

print("\n")
print("=" * 70)
print("BASELINE 1 — ONE-CLASS SVM")
print("=" * 70)

print(
    "\nTraining using:",
    X_train_scaled.shape[0],
    "BENIGN samples"
)

print(
    "Features:",
    len(ALL_FEATURES),
    "original /",
    X_train_scaled.shape[1],
    "encoded"
)


# ============================================================
# 6. TRAIN SVM
# ============================================================

svm_start = time.time()


svm_baseline = OneClassSVM(
    kernel=SVM_KERNEL,
    nu=SVM_NU
)


svm_baseline.fit(
    X_train_scaled
)


svm_train_time = (
    time.time() -
    svm_start
)


print(
    f"\nSVM training time: "
    f"{svm_train_time:.4f} seconds"
)


# ============================================================
# 7. SVM VALIDATION SCORES
# ============================================================
#
# IMPORTANT:
#
# We use the RAW decision_function directly.
#
# Your diagnostic showed:
#
#   Raw decision AUC  = 1.0
#   Negative raw AUC  = 0.0
#
# Therefore for this experiment:
#
#   larger raw decision score = more anomalous
#
# We do NOT negate it.
# ============================================================

svm_validation_start = time.time()


svm_validation_scores = (
    svm_baseline.decision_function(
        X_validation_scaled
    )
)


svm_validation_time = (
    time.time() -
    svm_validation_start
)


# ============================================================
# 8. VERIFY SVM SCORE DIRECTION
# ============================================================

svm_validation_auc_check = (
    roc_auc_score(
        y_validation,
        svm_validation_scores
    )
)


print(
    f"\nSVM validation score AUC: "
    f"{svm_validation_auc_check:.4f}"
)


if svm_validation_auc_check < 0.5:

    print(
        "\nWARNING:"
    )

    print(
        "SVM score direction appears reversed."
    )

    print(
        "Do not use this configuration for GA."
    )


# ============================================================
# 9. SVM THRESHOLD
# ============================================================
#
# ONLY benign validation samples are used.
# ============================================================

svm_validation_benign_scores = (
    svm_validation_scores[
        y_validation == 0
    ]
)


svm_threshold = np.percentile(
    svm_validation_benign_scores,
    THRESHOLD_PERCENTILE
)


print(
    f"SVM anomaly threshold: "
    f"{svm_threshold:.10f}"
)


# ============================================================
# 10. SVM FINAL TEST SCORES
# ============================================================

svm_final_start = time.time()


svm_final_scores = (
    svm_baseline.decision_function(
        X_final_scaled
    )
)


svm_prediction_time = (
    time.time() -
    svm_final_start
)


print(
    f"SVM final prediction time: "
    f"{svm_prediction_time:.4f} seconds"
)


# ============================================================
# 11. EVALUATE SVM
# ============================================================

svm_results = evaluate_anomaly_detector(

    model_name="FULL-FEATURE ONE-CLASS SVM",

    validation_scores=
        svm_validation_scores,

    final_scores=
        svm_final_scores,

    y_validation=
        y_validation,

    y_final=
        y_final,

    threshold=
        svm_threshold
)


svm_results["train_time"] = (
    svm_train_time
)

svm_results["prediction_time"] = (
    svm_validation_time +
    svm_prediction_time
)


# ============================================================
# 12. BASELINE 2 — KNN ANOMALY DETECTOR
# ============================================================

print("\n")
print("=" * 70)
print("BASELINE 2 — KNN ANOMALY DETECTOR")
print("=" * 70)

print(
    "\nK:",
    KNN_K
)

print(
    "Training samples:",
    X_train_scaled.shape[0]
)


# ============================================================
# 13. TRAIN KNN
# ============================================================
#
# KNN is fitted ONLY on benign training samples.
#
# There is essentially no expensive training phase;
# the computational cost occurs during nearest-neighbour
# queries.
# ============================================================

knn_start = time.time()


knn_baseline = NearestNeighbors(
    n_neighbors=KNN_K,
    algorithm="auto",
    n_jobs=-1
)


knn_baseline.fit(
    X_train_scaled
)


knn_train_time = (
    time.time() -
    knn_start
)


print(
    f"\nKNN training time: "
    f"{knn_train_time:.4f} seconds"
)


# ============================================================
# 14. KNN VALIDATION SCORES
# ============================================================
#
# Distance to K-th nearest benign neighbour:
#
# small distance = benign-like
# large distance = anomalous
# ============================================================

knn_validation_start = time.time()


validation_distances, _ = (
    knn_baseline.kneighbors(
        X_validation_scaled,
        n_neighbors=KNN_K
    )
)


knn_validation_scores = (
    validation_distances[:, -1]
)


knn_validation_time = (
    time.time() -
    knn_validation_start
)


# ============================================================
# 15. KNN THRESHOLD
# ============================================================

knn_validation_benign_scores = (
    knn_validation_scores[
        y_validation == 0
    ]
)


knn_threshold = np.percentile(
    knn_validation_benign_scores,
    THRESHOLD_PERCENTILE
)


print(
    f"\nKNN anomaly threshold: "
    f"{knn_threshold:.10f}"
)


# ============================================================
# 16. KNN FINAL TEST
# ============================================================

knn_final_start = time.time()


final_distances, _ = (
    knn_baseline.kneighbors(
        X_final_scaled,
        n_neighbors=KNN_K
    )
)


knn_final_scores = (
    final_distances[:, -1]
)


knn_prediction_time = (
    time.time() -
    knn_final_start
)


print(
    f"KNN validation prediction time: "
    f"{knn_validation_time:.4f} seconds"
)

print(
    f"KNN final prediction time: "
    f"{knn_prediction_time:.4f} seconds"
)


# ============================================================
# 17. EVALUATE KNN
# ============================================================

knn_results = evaluate_anomaly_detector(

    model_name="FULL-FEATURE KNN",

    validation_scores=
        knn_validation_scores,

    final_scores=
        knn_final_scores,

    y_validation=
        y_validation,

    y_final=
        y_final,

    threshold=
        knn_threshold
)


knn_results["train_time"] = (
    knn_train_time
)

knn_results["prediction_time"] = (
    knn_validation_time +
    knn_prediction_time
)


# ============================================================
# 18. BASELINE COMPARISON
# ============================================================

print("\n")
print("=" * 70)
print("PHASE B — BASELINE COMPARISON")
print("=" * 70)


print(
    f"\n{'Metric':<25}"
    f"{'SVM':>15}"
    f"{'KNN':>15}"
)

print("-" * 55)


print(
    f"{'Original features':<25}"
    f"{len(ALL_FEATURES):>15}"
    f"{len(ALL_FEATURES):>15}"
)


print(
    f"{'Encoded features':<25}"
    f"{X_train.shape[1]:>15}"
    f"{X_train.shape[1]:>15}"
)


print(
    f"{'Validation F1':<25}"
    f"{svm_results['val_f1']:>15.4f}"
    f"{knn_results['val_f1']:>15.4f}"
)


print(
    f"{'Validation ROC-AUC':<25}"
    f"{svm_results['val_auc']:>15.4f}"
    f"{knn_results['val_auc']:>15.4f}"
)


print(
    f"{'Final Accuracy':<25}"
    f"{svm_results['final_accuracy']:>15.4f}"
    f"{knn_results['final_accuracy']:>15.4f}"
)


print(
    f"{'Final Precision':<25}"
    f"{svm_results['final_precision']:>15.4f}"
    f"{knn_results['final_precision']:>15.4f}"
)


print(
    f"{'Final Recall':<25}"
    f"{svm_results['final_recall']:>15.4f}"
    f"{knn_results['final_recall']:>15.4f}"
)


print(
    f"{'Final F1':<25}"
    f"{svm_results['final_f1']:>15.4f}"
    f"{knn_results['final_f1']:>15.4f}"
)


print(
    f"{'Final ROC-AUC':<25}"
    f"{svm_results['final_auc']:>15.4f}"
    f"{knn_results['final_auc']:>15.4f}"
)


print(
    f"{'Training time (sec)':<25}"
    f"{svm_results['train_time']:>15.4f}"
    f"{knn_results['train_time']:>15.4f}"
)


print(
    f"{'Prediction time (sec)':<25}"
    f"{svm_results['prediction_time']:>15.4f}"
    f"{knn_results['prediction_time']:>15.4f}"
)


# ============================================================
# 19. SAVE BASELINE RESULTS
# ============================================================

baseline_results = {

    "svm": svm_results,

    "knn": knn_results
}


# ============================================================
# 20. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("PHASE B COMPLETE")
print("=" * 70)

print(
    "\nFull-feature baselines established:"
)

print(
    "\n  One-Class SVM"
)

print(
    f"    Final F1 : "
    f"{svm_results['final_f1']:.4f}"
)

print(
    f"    Final AUC: "
    f"{svm_results['final_auc']:.4f}"
)


print(
    "\n  KNN"
)

print(
    f"    Final F1 : "
    f"{knn_results['final_f1']:.4f}"
)

print(
    f"    Final AUC: "
    f"{knn_results['final_auc']:.4f}"
)


print(
    "\nThese are the full-feature baselines."
)

print(
    "Next phase:"
)

print(
    "  Binary GA + SVM"
)

print(
    "  Binary GA + KNN"
)

PHASE B — FULL-FEATURE BASELINES

Original features: 10
Encoded features: 16

Scaling features...
Scaling time: 0.0080 seconds
Training shape   : (10000, 16)
Validation shape : (5000, 16)
Final test shape : (100000, 16)


BASELINE 1 — ONE-CLASS SVM

Training using: 10000 BENIGN samples
Features: 10 original / 16 encoded

SVM training time: 0.1726 seconds

SVM validation score AUC: 1.0000
SVM anomaly threshold: 7287.8239394035
SVM final prediction time: 0.6934 seconds


FULL-FEATURE ONE-CLASS SVM — VALIDATION
Threshold : 7287.8239394035
Accuracy  : 0.9950
Precision : 0.9901
Recall    : 1.0000
F1        : 0.9950
ROC-AUC   : 1.0000

Confusion matrix:
[[2475   25]
 [   0 2500]]


FULL-FEATURE ONE-CLASS SVM — FINAL TEST
Accuracy  : 0.9953
Precision : 0.9907
Recall    : 1.0000
F1        : 0.9953
ROC-AUC   : 1.0000

Confusion matrix:
[[49532   468]
 [    0 50000]]


BASELINE 2 — KNN ANOMALY DETECTOR

K: 5
Training samples: 10000

KNN training time: 0.0010 seconds

KNN anomaly threshold: 0.013

In [11]:
# ============================================================
# PHASE C — BINARY GENETIC ALGORITHM FEATURE SELECTION
# ============================================================
#
# Goal:
#   Select a subset of the 10 ORIGINAL FEATURES.
#
# Classifiers:
#   1. One-Class SVM
#   2. KNN
#
# IMPORTANT:
#   - GA operates on ORIGINAL features.
#   - One-hot encoded columns belonging to a selected
#     categorical feature are selected together.
#   - Validation data is used for GA fitness.
#   - Final test data is NEVER used during GA evolution.
#   - Final test is evaluated only once using the best chromosome.
#
# Fitness:
#
#   Fitness = F1 - FEATURE_PENALTY * feature_ratio
#
# This encourages:
#   - High F1
#   - Fewer selected original features
#
# Runtime:
#   Small population + few generations.
# ============================================================

import time
import numpy as np

from sklearn.svm import OneClassSVM
from sklearn.neighbors import NearestNeighbors

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix
)


# ============================================================
# 1. GA CONFIGURATION
# ============================================================

GA_POPULATION_SIZE = 6
GA_GENERATIONS = 5

GA_MUTATION_RATE = 0.15
GA_CROSSOVER_RATE = 0.80

FEATURE_PENALTY = 0.01

RANDOM_SEED = 42

SVM_NU = 0.05
SVM_KERNEL = "linear"

KNN_K = 5

THRESHOLD_PERCENTILE = 99


# ============================================================
# 2. ORIGINAL FEATURES
# ============================================================

GA_FEATURES = [
    "id.orig_p",
    "id.resp_p",
    "duration",
    "orig_bytes",
    "resp_bytes",
    "orig_pkts",
    "resp_pkts",
    "proto",
    "conn_state",
    "history"
]

N_FEATURES = len(GA_FEATURES)


# ============================================================
# 3. MAP ORIGINAL FEATURES → ENCODED COLUMNS
# ============================================================
#
# X_train / X_validation / X_final contain 16 encoded columns.
#
# We need to know which encoded columns belong to each
# original feature.
#
# This assumes your Phase A one-hot encoder was created using
# the same GA_FEATURES order.
# ============================================================

print("=" * 70)
print("PHASE C — BINARY GENETIC ALGORITHM")
print("=" * 70)

print("\nOriginal features:")
for i, feature in enumerate(GA_FEATURES):
    print(f"  {i}: {feature}")


# ------------------------------------------------------------
# Find encoded column groups.
#
# These values correspond to your current Phase A output:
#
# id.orig_p       -> 1
# id.resp_p       -> 1
# duration        -> 1
# orig_bytes      -> 1
# resp_bytes      -> 1
# orig_pkts       -> 1
# resp_pkts       -> 1
# proto           -> 2
# conn_state      -> 3
# history         -> 4
#
# Total = 16 encoded columns.
# ------------------------------------------------------------

FEATURE_COLUMN_GROUPS = {

    "id.orig_p":
        list(range(0, 1)),

    "id.resp_p":
        list(range(1, 2)),

    "duration":
        list(range(2, 3)),

    "orig_bytes":
        list(range(3, 4)),

    "resp_bytes":
        list(range(4, 5)),

    "orig_pkts":
        list(range(5, 6)),

    "resp_pkts":
        list(range(6, 7)),

    "proto":
        list(range(7, 9)),

    "conn_state":
        list(range(9, 12)),

    "history":
        list(range(12, 16))
}


# ============================================================
# 4. RANDOM NUMBER GENERATOR
# ============================================================

rng = np.random.default_rng(
    RANDOM_SEED
)


# ============================================================
# 5. CHROMOSOME → ENCODED COLUMNS
# ============================================================

def chromosome_to_columns(chromosome):

    columns = []

    for bit, feature in zip(
        chromosome,
        GA_FEATURES
    ):

        if bit == 1:

            columns.extend(
                FEATURE_COLUMN_GROUPS[feature]
            )

    return columns


# ============================================================
# 6. CHROMOSOME → FEATURE NAMES
# ============================================================

def chromosome_to_features(chromosome):

    return [
        feature
        for bit, feature in zip(
            chromosome,
            GA_FEATURES
        )
        if bit == 1
    ]


# ============================================================
# 7. CREATE RANDOM CHROMOSOME
# ============================================================

def random_chromosome():

    chromosome = rng.integers(
        0,
        2,
        size=N_FEATURES
    )

    # Ensure at least one feature
    if chromosome.sum() == 0:

        chromosome[
            rng.integers(0, N_FEATURES)
        ] = 1

    return chromosome


# ============================================================
# 8. INITIAL POPULATION
# ============================================================

def create_population():

    return np.array([
        random_chromosome()
        for _ in range(
            GA_POPULATION_SIZE
        )
    ])


# ============================================================
# 9. TOURNAMENT SELECTION
# ============================================================

def tournament_selection(
    population,
    fitnesses,
    tournament_size=2
):

    indices = rng.choice(
        len(population),
        size=tournament_size,
        replace=False
    )

    best_index = indices[
        np.argmax(
            fitnesses[indices]
        )
    ]

    return population[
        best_index
    ].copy()


# ============================================================
# 10. SINGLE-POINT CROSSOVER
# ============================================================

def crossover(
    parent1,
    parent2
):

    if (
        rng.random()
        > GA_CROSSOVER_RATE
    ):

        return (
            parent1.copy(),
            parent2.copy()
        )

    point = rng.integers(
        1,
        N_FEATURES
    )

    child1 = np.concatenate([
        parent1[:point],
        parent2[point:]
    ])

    child2 = np.concatenate([
        parent2[:point],
        parent1[point:]
    ])

    return child1, child2


# ============================================================
# 11. BIT-FLIP MUTATION
# ============================================================

def mutate(chromosome):

    chromosome = chromosome.copy()

    for i in range(
        N_FEATURES
    ):

        if (
            rng.random()
            < GA_MUTATION_RATE
        ):

            chromosome[i] = (
                1 -
                chromosome[i]
            )

    # Never allow zero features

    if chromosome.sum() == 0:

        chromosome[
            rng.integers(0, N_FEATURES)
        ] = 1

    return chromosome


# ============================================================
# 12. SVM FITNESS
# ============================================================

def svm_fitness(
    chromosome
):

    selected_columns = (
        chromosome_to_columns(
            chromosome
        )
    )

    if len(selected_columns) == 0:
        return -1.0


    X_tr = X_train_scaled[
        :,
        selected_columns
    ]

    X_val = X_validation_scaled[
        :,
        selected_columns
    ]


    # --------------------------------------------------------
    # Train One-Class SVM
    # --------------------------------------------------------

    model = OneClassSVM(
        kernel=SVM_KERNEL,
        nu=SVM_NU
    )

    model.fit(
        X_tr
    )


    # --------------------------------------------------------
    # Decision scores
    #
    # IMPORTANT:
    # Use RAW decision_function because Phase B showed
    # this orientation gives ROC-AUC = 1.0.
    # --------------------------------------------------------

    scores = model.decision_function(
        X_val
    )


    # --------------------------------------------------------
    # Threshold from BENIGN validation only
    # --------------------------------------------------------

    benign_scores = scores[
        y_validation == 0
    ]

    threshold = np.percentile(
        benign_scores,
        THRESHOLD_PERCENTILE
    )


    predictions = (
        scores > threshold
    ).astype(np.int8)


    # --------------------------------------------------------
    # F1
    # --------------------------------------------------------

    f1 = f1_score(
        y_validation,
        predictions,
        zero_division=0
    )


    # --------------------------------------------------------
    # Feature penalty
    # --------------------------------------------------------

    feature_ratio = (
        chromosome.sum()
        / N_FEATURES
    )


    fitness = (
        f1
        - FEATURE_PENALTY *
        feature_ratio
    )


    return fitness


# ============================================================
# 13. KNN FITNESS
# ============================================================

def knn_fitness(
    chromosome
):

    selected_columns = (
        chromosome_to_columns(
            chromosome
        )
    )

    if len(selected_columns) == 0:
        return -1.0


    X_tr = X_train_scaled[
        :,
        selected_columns
    ]

    X_val = X_validation_scaled[
        :,
        selected_columns
    ]


    # --------------------------------------------------------
    # KNN
    # --------------------------------------------------------

    model = NearestNeighbors(
        n_neighbors=KNN_K,
        algorithm="auto",
        n_jobs=-1
    )

    model.fit(
        X_tr
    )


    distances, _ = model.kneighbors(
        X_val,
        n_neighbors=KNN_K
    )


    scores = distances[:, -1]


    # --------------------------------------------------------
    # Threshold from BENIGN validation only
    # --------------------------------------------------------

    benign_scores = scores[
        y_validation == 0
    ]

    threshold = np.percentile(
        benign_scores,
        THRESHOLD_PERCENTILE
    )


    predictions = (
        scores > threshold
    ).astype(np.int8)


    # --------------------------------------------------------
    # F1
    # --------------------------------------------------------

    f1 = f1_score(
        y_validation,
        predictions,
        zero_division=0
    )


    # --------------------------------------------------------
    # Feature penalty
    # --------------------------------------------------------

    feature_ratio = (
        chromosome.sum()
        / N_FEATURES
    )


    fitness = (
        f1
        - FEATURE_PENALTY *
        feature_ratio
    )


    return fitness


# ============================================================
# 14. GENERIC BINARY GA
# ============================================================

def run_binary_ga(
    fitness_function,
    classifier_name
):

    print("\n")
    print("=" * 70)
    print(
        f"BINARY GA + {classifier_name}"
    )
    print("=" * 70)

    print(
        f"\nPopulation size : "
        f"{GA_POPULATION_SIZE}"
    )

    print(
        f"Generations     : "
        f"{GA_GENERATIONS}"
    )

    print(
        f"Mutation rate   : "
        f"{GA_MUTATION_RATE}"
    )

    print(
        f"Crossover rate  : "
        f"{GA_CROSSOVER_RATE}"
    )

    print(
        f"Feature penalty : "
        f"{FEATURE_PENALTY}"
    )


    # --------------------------------------------------------
    # Start timer
    # --------------------------------------------------------

    start_time = time.time()


    # --------------------------------------------------------
    # Create population
    # --------------------------------------------------------

    population = (
        create_population()
    )


    best_chromosome = None
    best_fitness = -np.inf

    fitness_history = []


    # ========================================================
    # GENERATIONS
    # ========================================================

    for generation in range(
        GA_GENERATIONS
    ):

        fitnesses = np.array([

            fitness_function(
                chromosome
            )

            for chromosome
            in population

        ])


        # ----------------------------------------------------
        # Find generation best
        # ----------------------------------------------------

        generation_best_index = (
            np.argmax(fitnesses)
        )

        generation_best_fitness = (
            fitnesses[
                generation_best_index
            ]
        )

        generation_best = (
            population[
                generation_best_index
            ].copy()
        )


        # ----------------------------------------------------
        # Global best
        # ----------------------------------------------------

        if (
            generation_best_fitness
            > best_fitness
        ):

            best_fitness = (
                generation_best_fitness
            )

            best_chromosome = (
                generation_best.copy()
            )


        fitness_history.append(
            best_fitness
        )


        # ----------------------------------------------------
        # Print progress
        # ----------------------------------------------------

        print(
            f"Generation "
            f"{generation + 1}/"
            f"{GA_GENERATIONS}"
            f" | Best fitness: "
            f"{best_fitness:.4f}"
            f" | Selected features: "
            f"{best_chromosome.sum()}"
        )


        # ----------------------------------------------------
        # Elitism
        # ----------------------------------------------------

        new_population = [
            best_chromosome.copy()
        ]


        # ----------------------------------------------------
        # Generate offspring
        # ----------------------------------------------------

        while len(
            new_population
        ) < GA_POPULATION_SIZE:

            parent1 = (
                tournament_selection(
                    population,
                    fitnesses
                )
            )

            parent2 = (
                tournament_selection(
                    population,
                    fitnesses
                )
            )


            child1, child2 = (
                crossover(
                    parent1,
                    parent2
                )
            )


            child1 = mutate(
                child1
            )

            child2 = mutate(
                child2
            )


            new_population.append(
                child1
            )


            if len(
                new_population
            ) < GA_POPULATION_SIZE:

                new_population.append(
                    child2
                )


        population = np.array(
            new_population
        )


    # ========================================================
    # END TIMER
    # ========================================================

    runtime = (
        time.time()
        - start_time
    )


    # ========================================================
    # PRINT RESULT
    # ========================================================

    print("\n")
    print("=" * 70)
    print(
        f"{classifier_name} GA COMPLETE"
    )
    print("=" * 70)

    print(
        f"GA runtime: "
        f"{runtime:.2f} seconds "
        f"({runtime / 60:.2f} minutes)"
    )

    print(
        f"Best fitness: "
        f"{best_fitness:.4f}"
    )

    print(
        f"Selected features: "
        f"{best_chromosome.sum()} / "
        f"{N_FEATURES}"
    )


    print(
        "\nBest chromosome:"
    )

    print(
        best_chromosome
    )


    selected_features = (
        chromosome_to_features(
            best_chromosome
        )
    )

    removed_features = [
        feature
        for feature in GA_FEATURES
        if feature
        not in selected_features
    ]


    print(
        "\nSelected features:"
    )

    for feature in selected_features:

        print(
            f"  + {feature}"
        )


    print(
        "\nRemoved features:"
    )

    for feature in removed_features:

        print(
            f"  - {feature}"
        )


    selected_columns = (
        chromosome_to_columns(
            best_chromosome
        )
    )


    print(
        "\nEncoded columns selected:",
        len(selected_columns)
    )


    return {

        "classifier":
            classifier_name,

        "chromosome":
            best_chromosome,

        "fitness":
            best_fitness,

        "selected_features":
            selected_features,

        "removed_features":
            removed_features,

        "selected_columns":
            selected_columns,

        "fitness_history":
            fitness_history,

        "runtime":
            runtime
    }


# ============================================================
# 15. RUN BINARY GA — SVM
# ============================================================

ga_svm_results = run_binary_ga(
    svm_fitness,
    "ONE-CLASS SVM"
)


# ============================================================
# 16. RUN BINARY GA — KNN
# ============================================================

ga_knn_results = run_binary_ga(
    knn_fitness,
    "KNN"
)


# ============================================================
# 17. SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("PHASE C — BINARY GA SUMMARY")
print("=" * 70)


print("\nONE-CLASS SVM")

print(
    f"Fitness: "
    f"{ga_svm_results['fitness']:.4f}"
)

print(
    f"Features selected: "
    f"{len(ga_svm_results['selected_features'])}"
)

print(
    "Selected:"
)

for feature in (
    ga_svm_results[
        "selected_features"
    ]
):

    print(
        f"  + {feature}"
    )


print("\nKNN")

print(
    f"Fitness: "
    f"{ga_knn_results['fitness']:.4f}"
)

print(
    f"Features selected: "
    f"{len(ga_knn_results['selected_features'])}"
)

print(
    "Selected:"
)

for feature in (
    ga_knn_results[
        "selected_features"
    ]
):

    print(
        f"  + {feature}"
    )


print("\n")
print("=" * 70)
print("PHASE C COMPLETE")
print("=" * 70)

print(
    "\nIMPORTANT:"
)

print(
    "The FINAL TEST has NOT been used by the GA."
)

print(
    "The selected feature subsets will now be"
)

print(
    "evaluated on the final test set exactly once."
)

PHASE C — BINARY GENETIC ALGORITHM

Original features:
  0: id.orig_p
  1: id.resp_p
  2: duration
  3: orig_bytes
  4: resp_bytes
  5: orig_pkts
  6: resp_pkts
  7: proto
  8: conn_state
  9: history


BINARY GA + ONE-CLASS SVM

Population size : 6
Generations     : 5
Mutation rate   : 0.15
Crossover rate  : 0.8
Feature penalty : 0.01
Generation 1/5 | Best fitness: 0.9910 | Selected features: 4
Generation 2/5 | Best fitness: 0.9910 | Selected features: 4
Generation 3/5 | Best fitness: 0.9970 | Selected features: 3
Generation 4/5 | Best fitness: 0.9970 | Selected features: 3
Generation 5/5 | Best fitness: 0.9970 | Selected features: 3


ONE-CLASS SVM GA COMPLETE
GA runtime: 3.93 seconds (0.07 minutes)
Best fitness: 0.9970
Selected features: 3 / 10

Best chromosome:
[0 1 1 0 0 1 0 0 0 0]

Selected features:
  + id.resp_p
  + duration
  + orig_pkts

Removed features:
  - id.orig_p
  - orig_bytes
  - resp_bytes
  - resp_pkts
  - proto
  - conn_state
  - history

Encoded columns selected: 

In [12]:
# ============================================================
# PHASE C.1 — BINARY GA FINAL TEST EVALUATION
# ============================================================
#
# Uses the BEST chromosomes found in Phase C.
#
# SVM:
#   id.resp_p
#   duration
#   orig_pkts
#
# KNN:
#   id.resp_p
#   resp_bytes
#
# IMPORTANT:
#   - GA is already finished.
#   - No feature selection is performed here.
#   - Validation is NOT used to change the selected features.
#   - Final test is evaluated once.
#   - Same scaling and threshold methodology as Phase B.
# ============================================================

import time
import numpy as np

from sklearn.svm import OneClassSVM
from sklearn.neighbors import NearestNeighbors

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


print("=" * 70)
print("PHASE C.1 — BINARY GA FINAL TEST EVALUATION")
print("=" * 70)


# ============================================================
# 1. GET BEST CHROMOSOMES FROM PHASE C
# ============================================================

svm_chromosome = (
    ga_svm_results["chromosome"]
)

knn_chromosome = (
    ga_knn_results["chromosome"]
)


# ============================================================
# 2. GET SELECTED ENCODED COLUMNS
# ============================================================

svm_selected_columns = (
    chromosome_to_columns(
        svm_chromosome
    )
)

knn_selected_columns = (
    chromosome_to_columns(
        knn_chromosome
    )
)


svm_selected_features = (
    chromosome_to_features(
        svm_chromosome
    )
)

knn_selected_features = (
    chromosome_to_features(
        knn_chromosome
    )
)


# ============================================================
# 3. PRINT SELECTED FEATURES
# ============================================================

print("\nBinary GA — SVM")

print(
    "Selected original features:"
)

for feature in svm_selected_features:
    print(f"  + {feature}")

print(
    f"Original features: "
    f"{len(svm_selected_features)} / "
    f"{N_FEATURES}"
)

print(
    f"Encoded columns: "
    f"{len(svm_selected_columns)}"
)


print("\nBinary GA — KNN")

print(
    "Selected original features:"
)

for feature in knn_selected_features:
    print(f"  + {feature}")

print(
    f"Original features: "
    f"{len(knn_selected_features)} / "
    f"{N_FEATURES}"
)

print(
    f"Encoded columns: "
    f"{len(knn_selected_columns)}"
)


# ============================================================
# 4. CREATE GA-SELECTED DATASETS
# ============================================================

X_train_svm_ga = (
    X_train_scaled[
        :,
        svm_selected_columns
    ]
)

X_validation_svm_ga = (
    X_validation_scaled[
        :,
        svm_selected_columns
    ]
)

X_final_svm_ga = (
    X_final_scaled[
        :,
        svm_selected_columns
    ]
)


X_train_knn_ga = (
    X_train_scaled[
        :,
        knn_selected_columns
    ]
)

X_validation_knn_ga = (
    X_validation_scaled[
        :,
        knn_selected_columns
    ]
)

X_final_knn_ga = (
    X_final_scaled[
        :,
        knn_selected_columns
    ]
)


# ============================================================
# 5. GA + SVM
# ============================================================

print("\n")
print("=" * 70)
print("BINARY GA + ONE-CLASS SVM — FINAL TEST")
print("=" * 70)


print(
    "\nTraining shape:",
    X_train_svm_ga.shape
)

print(
    "Validation shape:",
    X_validation_svm_ga.shape
)

print(
    "Final test shape:",
    X_final_svm_ga.shape
)


# ============================================================
# 6. TRAIN SVM
# ============================================================

svm_ga_train_start = time.time()


svm_ga_model = OneClassSVM(
    kernel=SVM_KERNEL,
    nu=SVM_NU
)


svm_ga_model.fit(
    X_train_svm_ga
)


svm_ga_train_time = (
    time.time()
    - svm_ga_train_start
)


print(
    f"\nTraining time: "
    f"{svm_ga_train_time:.4f} seconds"
)


# ============================================================
# 7. VALIDATION SCORES
# ============================================================

svm_ga_val_start = time.time()


svm_ga_validation_scores = (
    svm_ga_model.decision_function(
        X_validation_svm_ga
    )
)


svm_ga_validation_time = (
    time.time()
    - svm_ga_val_start
)


# ============================================================
# 8. VALIDATION THRESHOLD
# ============================================================
#
# IMPORTANT:
# Threshold is calculated from BENIGN validation samples.
# Exactly the same methodology as Phase B.
# ============================================================

svm_ga_benign_scores = (
    svm_ga_validation_scores[
        y_validation == 0
    ]
)


svm_ga_threshold = np.percentile(
    svm_ga_benign_scores,
    THRESHOLD_PERCENTILE
)


# ============================================================
# 9. VALIDATION PREDICTIONS
# ============================================================

svm_ga_validation_predictions = (
    svm_ga_validation_scores
    > svm_ga_threshold
).astype(np.int8)


# ============================================================
# 10. VALIDATION METRICS
# ============================================================

svm_ga_val_accuracy = accuracy_score(
    y_validation,
    svm_ga_validation_predictions
)

svm_ga_val_precision = precision_score(
    y_validation,
    svm_ga_validation_predictions,
    zero_division=0
)

svm_ga_val_recall = recall_score(
    y_validation,
    svm_ga_validation_predictions,
    zero_division=0
)

svm_ga_val_f1 = f1_score(
    y_validation,
    svm_ga_validation_predictions,
    zero_division=0
)

svm_ga_val_auc = roc_auc_score(
    y_validation,
    svm_ga_validation_scores
)

svm_ga_val_cm = confusion_matrix(
    y_validation,
    svm_ga_validation_predictions
)


# ============================================================
# 11. FINAL TEST PREDICTION
# ============================================================

svm_ga_final_start = time.time()


svm_ga_final_scores = (
    svm_ga_model.decision_function(
        X_final_svm_ga
    )
)


svm_ga_final_prediction_time = (
    time.time()
    - svm_ga_final_start
)


# ============================================================
# 12. FINAL TEST PREDICTIONS
# ============================================================

svm_ga_final_predictions = (
    svm_ga_final_scores
    > svm_ga_threshold
).astype(np.int8)


# ============================================================
# 13. FINAL TEST METRICS
# ============================================================

svm_ga_final_accuracy = accuracy_score(
    y_final,
    svm_ga_final_predictions
)

svm_ga_final_precision = precision_score(
    y_final,
    svm_ga_final_predictions,
    zero_division=0
)

svm_ga_final_recall = recall_score(
    y_final,
    svm_ga_final_predictions,
    zero_division=0
)

svm_ga_final_f1 = f1_score(
    y_final,
    svm_ga_final_predictions,
    zero_division=0
)

svm_ga_final_auc = roc_auc_score(
    y_final,
    svm_ga_final_scores
)

svm_ga_final_cm = confusion_matrix(
    y_final,
    svm_ga_final_predictions
)


# ============================================================
# 14. PRINT SVM RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("BINARY GA + ONE-CLASS SVM — VALIDATION")
print("=" * 70)

print(
    f"Threshold : "
    f"{svm_ga_threshold:.10f}"
)

print(
    f"Accuracy  : "
    f"{svm_ga_val_accuracy:.4f}"
)

print(
    f"Precision : "
    f"{svm_ga_val_precision:.4f}"
)

print(
    f"Recall    : "
    f"{svm_ga_val_recall:.4f}"
)

print(
    f"F1        : "
    f"{svm_ga_val_f1:.4f}"
)

print(
    f"ROC-AUC   : "
    f"{svm_ga_val_auc:.4f}"
)

print("\nConfusion matrix:")

print(
    svm_ga_val_cm
)


print("\n")
print("=" * 70)
print("BINARY GA + ONE-CLASS SVM — FINAL TEST")
print("=" * 70)

print(
    f"Accuracy  : "
    f"{svm_ga_final_accuracy:.4f}"
)

print(
    f"Precision : "
    f"{svm_ga_final_precision:.4f}"
)

print(
    f"Recall    : "
    f"{svm_ga_final_recall:.4f}"
)

print(
    f"F1        : "
    f"{svm_ga_final_f1:.4f}"
)

print(
    f"ROC-AUC   : "
    f"{svm_ga_final_auc:.4f}"
)

print("\nConfusion matrix:")

print(
    svm_ga_final_cm
)

print(
    f"\nFinal prediction time: "
    f"{svm_ga_final_prediction_time:.4f} seconds"
)


# ============================================================
# 15. GA + KNN
# ============================================================

print("\n")
print("=" * 70)
print("BINARY GA + KNN — FINAL TEST")
print("=" * 70)


print(
    "\nTraining shape:",
    X_train_knn_ga.shape
)

print(
    "Validation shape:",
    X_validation_knn_ga.shape
)

print(
    "Final test shape:",
    X_final_knn_ga.shape
)


# ============================================================
# 16. TRAIN KNN
# ============================================================

knn_ga_train_start = time.time()


knn_ga_model = NearestNeighbors(
    n_neighbors=KNN_K,
    algorithm="auto",
    n_jobs=-1
)


knn_ga_model.fit(
    X_train_knn_ga
)


knn_ga_train_time = (
    time.time()
    - knn_ga_train_start
)


print(
    f"\nTraining time: "
    f"{knn_ga_train_time:.4f} seconds"
)


# ============================================================
# 17. VALIDATION SCORES
# ============================================================

knn_ga_val_start = time.time()


knn_ga_validation_distances, _ = (
    knn_ga_model.kneighbors(
        X_validation_knn_ga,
        n_neighbors=KNN_K
    )
)


knn_ga_validation_scores = (
    knn_ga_validation_distances[:, -1]
)


knn_ga_validation_time = (
    time.time()
    - knn_ga_val_start
)


# ============================================================
# 18. VALIDATION THRESHOLD
# ============================================================

knn_ga_benign_scores = (
    knn_ga_validation_scores[
        y_validation == 0
    ]
)


knn_ga_threshold = np.percentile(
    knn_ga_benign_scores,
    THRESHOLD_PERCENTILE
)


# ============================================================
# 19. VALIDATION PREDICTIONS
# ============================================================

knn_ga_validation_predictions = (
    knn_ga_validation_scores
    > knn_ga_threshold
).astype(np.int8)


# ============================================================
# 20. VALIDATION METRICS
# ============================================================

knn_ga_val_accuracy = accuracy_score(
    y_validation,
    knn_ga_validation_predictions
)

knn_ga_val_precision = precision_score(
    y_validation,
    knn_ga_validation_predictions,
    zero_division=0
)

knn_ga_val_recall = recall_score(
    y_validation,
    knn_ga_validation_predictions,
    zero_division=0
)

knn_ga_val_f1 = f1_score(
    y_validation,
    knn_ga_validation_predictions,
    zero_division=0
)

knn_ga_val_auc = roc_auc_score(
    y_validation,
    knn_ga_validation_scores
)

knn_ga_val_cm = confusion_matrix(
    y_validation,
    knn_ga_validation_predictions
)


# ============================================================
# 21. FINAL TEST PREDICTION
# ============================================================

knn_ga_final_start = time.time()


knn_ga_final_distances, _ = (
    knn_ga_model.kneighbors(
        X_final_knn_ga,
        n_neighbors=KNN_K
    )
)


knn_ga_final_scores = (
    knn_ga_final_distances[:, -1]
)


knn_ga_final_prediction_time = (
    time.time()
    - knn_ga_final_start
)


# ============================================================
# 22. FINAL TEST PREDICTIONS
# ============================================================

knn_ga_final_predictions = (
    knn_ga_final_scores
    > knn_ga_threshold
).astype(np.int8)


# ============================================================
# 23. FINAL TEST METRICS
# ============================================================

knn_ga_final_accuracy = accuracy_score(
    y_final,
    knn_ga_final_predictions
)

knn_ga_final_precision = precision_score(
    y_final,
    knn_ga_final_predictions,
    zero_division=0
)

knn_ga_final_recall = recall_score(
    y_final,
    knn_ga_final_predictions,
    zero_division=0
)

knn_ga_final_f1 = f1_score(
    y_final,
    knn_ga_final_predictions,
    zero_division=0
)

knn_ga_final_auc = roc_auc_score(
    y_final,
    knn_ga_final_scores
)

knn_ga_final_cm = confusion_matrix(
    y_final,
    knn_ga_final_predictions
)


# ============================================================
# 24. PRINT KNN RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("BINARY GA + KNN — VALIDATION")
print("=" * 70)

print(
    f"Threshold : "
    f"{knn_ga_threshold:.10f}"
)

print(
    f"Accuracy  : "
    f"{knn_ga_val_accuracy:.4f}"
)

print(
    f"Precision : "
    f"{knn_ga_val_precision:.4f}"
)

print(
    f"Recall    : "
    f"{knn_ga_val_recall:.4f}"
)

print(
    f"F1        : "
    f"{knn_ga_val_f1:.4f}"
)

print(
    f"ROC-AUC   : "
    f"{knn_ga_val_auc:.4f}"
)

print("\nConfusion matrix:")

print(
    knn_ga_val_cm
)


print("\n")
print("=" * 70)
print("BINARY GA + KNN — FINAL TEST")
print("=" * 70)

print(
    f"Accuracy  : "
    f"{knn_ga_final_accuracy:.4f}"
)

print(
    f"Precision : "
    f"{knn_ga_final_precision:.4f}"
)

print(
    f"Recall    : "
    f"{knn_ga_final_recall:.4f}"
)

print(
    f"F1        : "
    f"{knn_ga_final_f1:.4f}"
)

print(
    f"ROC-AUC   : "
    f"{knn_ga_final_auc:.4f}"
)

print("\nConfusion matrix:")

print(
    knn_ga_final_cm
)

print(
    f"\nFinal prediction time: "
    f"{knn_ga_final_prediction_time:.4f} seconds"
)


# ============================================================
# 25. FINAL COMPARISON
# ============================================================

print("\n")
print("=" * 70)
print("PHASE C — FULL FEATURES vs BINARY GA")
print("=" * 70)


print(
    f"\n{'Metric':<30}"
    f"{'SVM Full':>12}"
    f"{'SVM GA':>12}"
    f"{'KNN Full':>12}"
    f"{'KNN GA':>12}"
)

print("-" * 78)


print(
    f"{'Original features':<30}"
    f"{10:>12}"
    f"{len(svm_selected_features):>12}"
    f"{10:>12}"
    f"{len(knn_selected_features):>12}"
)


print(
    f"{'Encoded features':<30}"
    f"{16:>12}"
    f"{len(svm_selected_columns):>12}"
    f"{16:>12}"
    f"{len(knn_selected_columns):>12}"
)


print(
    f"{'Final Accuracy':<30}"
    f"{svm_results['final_accuracy']:>12.4f}"
    f"{svm_ga_final_accuracy:>12.4f}"
    f"{knn_results['final_accuracy']:>12.4f}"
    f"{knn_ga_final_accuracy:>12.4f}"
)


print(
    f"{'Final Precision':<30}"
    f"{svm_results['final_precision']:>12.4f}"
    f"{svm_ga_final_precision:>12.4f}"
    f"{knn_results['final_precision']:>12.4f}"
    f"{knn_ga_final_precision:>12.4f}"
)


print(
    f"{'Final Recall':<30}"
    f"{svm_results['final_recall']:>12.4f}"
    f"{svm_ga_final_recall:>12.4f}"
    f"{knn_results['final_recall']:>12.4f}"
    f"{knn_ga_final_recall:>12.4f}"
)


print(
    f"{'Final F1':<30}"
    f"{svm_results['final_f1']:>12.4f}"
    f"{svm_ga_final_f1:>12.4f}"
    f"{knn_results['final_f1']:>12.4f}"
    f"{knn_ga_final_f1:>12.4f}"
)


print(
    f"{'Final ROC-AUC':<30}"
    f"{svm_results['final_auc']:>12.4f}"
    f"{svm_ga_final_auc:>12.4f}"
    f"{knn_results['final_auc']:>12.4f}"
    f"{knn_ga_final_auc:>12.4f}"
)


# ============================================================
# 26. FEATURE REDUCTION
# ============================================================

svm_reduction = (
    1 -
    len(svm_selected_features) / 10
) * 100


knn_reduction = (
    1 -
    len(knn_selected_features) / 10
) * 100


print("\n")
print("=" * 70)
print("FEATURE REDUCTION")
print("=" * 70)

print(
    f"\nSVM:"
)

print(
    f"  10 → "
    f"{len(svm_selected_features)} features"
)

print(
    f"  Reduction: "
    f"{svm_reduction:.1f}%"
)


print(
    f"\nKNN:"
)

print(
    f"  10 → "
    f"{len(knn_selected_features)} features"
)

print(
    f"  Reduction: "
    f"{knn_reduction:.1f}%"
)


# ============================================================
# 27. STORE RESULTS
# ============================================================

phase_c_final_results = {

    "svm": {

        "selected_features":
            svm_selected_features,

        "selected_columns":
            svm_selected_columns,

        "n_features":
            len(svm_selected_features),

        "accuracy":
            svm_ga_final_accuracy,

        "precision":
            svm_ga_final_precision,

        "recall":
            svm_ga_final_recall,

        "f1":
            svm_ga_final_f1,

        "auc":
            svm_ga_final_auc,

        "confusion_matrix":
            svm_ga_final_cm,

        "train_time":
            svm_ga_train_time,

        "prediction_time":
            svm_ga_final_prediction_time
    },

    "knn": {

        "selected_features":
            knn_selected_features,

        "selected_columns":
            knn_selected_columns,

        "n_features":
            len(knn_selected_features),

        "accuracy":
            knn_ga_final_accuracy,

        "precision":
            knn_ga_final_precision,

        "recall":
            knn_ga_final_recall,

        "f1":
            knn_ga_final_f1,

        "auc":
            knn_ga_final_auc,

        "confusion_matrix":
            knn_ga_final_cm,

        "train_time":
            knn_ga_train_time,

        "prediction_time":
            knn_ga_final_prediction_time
    }
}


# ============================================================
# 28. COMPLETE
# ============================================================

print("\n")
print("=" * 70)
print("PHASE C.1 COMPLETE")
print("=" * 70)

print(
    "\nBinary GA selected features have now been"
)

print(
    "evaluated on the untouched 100,000-row final test set."
)

print(
    "\nNext phase:"
)

print(
    "  Phase D — 1+1 Evolutionary Strategy"
)

PHASE C.1 — BINARY GA FINAL TEST EVALUATION

Binary GA — SVM
Selected original features:
  + id.resp_p
  + duration
  + orig_pkts
Original features: 3 / 10
Encoded columns: 3

Binary GA — KNN
Selected original features:
  + id.resp_p
  + resp_bytes
Original features: 2 / 10
Encoded columns: 2


BINARY GA + ONE-CLASS SVM — FINAL TEST

Training shape: (10000, 3)
Validation shape: (5000, 3)
Final test shape: (100000, 3)

Training time: 0.0755 seconds


BINARY GA + ONE-CLASS SVM — VALIDATION
Threshold : 1003.5334540229
Accuracy  : 1.0000
Precision : 1.0000
Recall    : 1.0000
F1        : 1.0000
ROC-AUC   : 1.0000

Confusion matrix:
[[2500    0]
 [   0 2500]]


BINARY GA + ONE-CLASS SVM — FINAL TEST
Accuracy  : 1.0000
Precision : 0.9999
Recall    : 1.0000
F1        : 1.0000
ROC-AUC   : 1.0000

Confusion matrix:
[[49995     5]
 [    0 50000]]

Final prediction time: 0.3775 seconds


BINARY GA + KNN — FINAL TEST

Training shape: (10000, 2)
Validation shape: (5000, 2)
Final test shape: (100000,

In [14]:
# ============================================================
# PHASE D — (1+1)-EVOLUTIONARY STRATEGY
# ============================================================
#
# Goal:
#   Select a subset of the 10 ORIGINAL FEATURES using a
#   binary (1+1)-Evolutionary Strategy.
#
# Classifiers:
#   1. One-Class SVM
#   2. KNN
#
# IMPORTANT:
#   - ES operates on ORIGINAL features.
#   - One-hot encoded columns belonging to a selected
#     categorical feature are selected together.
#   - Validation data is used for fitness.
#   - FINAL TEST is NEVER used during evolution.
#   - Final test is evaluated only once using the best
#     chromosome.
#
# Algorithm:
#
#   1. Create one parent chromosome.
#   2. Evaluate parent.
#   3. Mutate parent to create one offspring.
#   4. Evaluate offspring.
#   5. If offspring fitness >= parent fitness:
#          offspring becomes new parent
#      else:
#          keep parent
#   6. Repeat for N iterations.
#
# Mutation:
#
#   Bit flip probability = 1 / number_of_features
#
# Fitness:
#
#   Fitness = F1 - FEATURE_PENALTY * feature_ratio
#
# ============================================================


import time
import numpy as np

from sklearn.svm import OneClassSVM
from sklearn.neighbors import NearestNeighbors

from sklearn.metrics import (
    f1_score,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix
)


# ============================================================
# 1. CONFIGURATION
# ============================================================

ES_ITERATIONS = 50

ES_MUTATION_RATE = 1.0 / len(ALL_FEATURES)

FEATURE_PENALTY = 0.01

ES_RANDOM_SEED = 42

SVM_NU = 0.05
SVM_KERNEL = "linear"

KNN_K = 5

THRESHOLD_PERCENTILE = 99


# ============================================================
# 2. ORIGINAL FEATURES
# ============================================================
#
# IMPORTANT:
#
# We use ALL_FEATURES from Phase A.
#
# Do NOT use ORIGINAL_FEATURES here.
# ============================================================

ES_FEATURES = ALL_FEATURES.copy()

N_ES_FEATURES = len(ES_FEATURES)


# ============================================================
# 3. VERIFY FEATURE COUNT
# ============================================================

print("=" * 70)
print("PHASE D — (1+1)-EVOLUTIONARY STRATEGY")
print("=" * 70)

print("\nOriginal features:")

for i, feature in enumerate(ES_FEATURES):

    print(
        f"  {i}: {feature}"
    )


print(
    "\nTotal original features:",
    N_ES_FEATURES
)

print(
    "Mutation probability:",
    f"{ES_MUTATION_RATE:.4f}"
)

print(
    "Iterations:",
    ES_ITERATIONS
)

print(
    "Feature penalty:",
    FEATURE_PENALTY
)


# ============================================================
# 4. VERIFY FEATURE MAPPING
# ============================================================
#
# Phase A already created feature_groups.
#
# feature_groups maps:
#
#   original feature
#       ↓
#   encoded column indices
#
# Example:
#
#   proto -> [7, 8]
#   conn_state -> [9, 10, 11]
#   history -> [12, 13, 14, 15]
#
# This means categorical features are selected as a group.
# ============================================================

print("\n")
print("=" * 70)
print("ES FEATURE → ENCODED COLUMN MAPPING")
print("=" * 70)


for feature in ES_FEATURES:

    print(
        f"{feature:<15} -> "
        f"{feature_groups[feature]}"
    )


# ============================================================
# 5. RANDOM NUMBER GENERATOR
# ============================================================

es_rng = np.random.default_rng(
    ES_RANDOM_SEED
)


# ============================================================
# 6. CHROMOSOME → ENCODED COLUMNS
# ============================================================

def es_chromosome_to_columns(
    chromosome
):

    selected_columns = []

    for bit, feature in zip(
        chromosome,
        ES_FEATURES
    ):

        if bit == 1:

            selected_columns.extend(
                feature_groups[feature]
            )

    return sorted(
        selected_columns
    )


# ============================================================
# 7. CHROMOSOME → FEATURE NAMES
# ============================================================

def es_chromosome_to_features(
    chromosome
):

    return [

        feature

        for bit, feature in zip(
            chromosome,
            ES_FEATURES
        )

        if bit == 1
    ]


# ============================================================
# 8. CREATE INITIAL CHROMOSOME
# ============================================================
#
# The initial solution contains all features.
#
# This is useful because the ES begins from the same
# full-feature search space as the baseline.
#
# We could also start randomly, but starting from the
# full feature set gives a clean feature-elimination
# interpretation.
# ============================================================

def create_es_initial_chromosome():

    chromosome = np.ones(
        N_ES_FEATURES,
        dtype=np.int8
    )

    return chromosome


# ============================================================
# 9. MUTATION
# ============================================================
#
# Binary (1+1)-ES mutation:
#
# Each bit is flipped independently with probability:
#
#       1 / n
#
# where n = number of features.
#
# With 10 features:
#
#       p = 0.1
#
# This gives approximately one changed bit per mutation.
#
# We also prevent the zero-feature solution.
# ============================================================

def mutate_es(
    chromosome
):

    offspring = chromosome.copy()


    # --------------------------------------------------------
    # Bit-flip mutation
    # --------------------------------------------------------

    mutation_mask = (
        es_rng.random(
            N_ES_FEATURES
        )
        < ES_MUTATION_RATE
    )


    offspring[
        mutation_mask
    ] = (
        1 -
        offspring[
            mutation_mask
        ]
    )


    # --------------------------------------------------------
    # Prevent zero selected features
    # --------------------------------------------------------

    if offspring.sum() == 0:

        offspring[
            es_rng.integers(
                0,
                N_ES_FEATURES
            )
        ] = 1


    return offspring


# ============================================================
# 10. SVM FITNESS
# ============================================================

def es_svm_fitness(
    chromosome
):

    selected_columns = (
        es_chromosome_to_columns(
            chromosome
        )
    )


    # --------------------------------------------------------
    # Safety check
    # --------------------------------------------------------

    if len(selected_columns) == 0:

        return -1.0


    # --------------------------------------------------------
    # Select encoded columns
    # --------------------------------------------------------

    X_tr = X_train_scaled[
        :,
        selected_columns
    ]

    X_val = X_validation_scaled[
        :,
        selected_columns
    ]


    # --------------------------------------------------------
    # Train One-Class SVM
    # --------------------------------------------------------

    model = OneClassSVM(

        kernel=SVM_KERNEL,

        nu=SVM_NU
    )


    model.fit(
        X_tr
    )


    # --------------------------------------------------------
    # Validation anomaly scores
    # --------------------------------------------------------
    #
    # IMPORTANT:
    #
    # We retain the same score orientation used in
    # Phase B and Phase C.
    #
    # Larger score = more anomalous.
    # --------------------------------------------------------

    scores = model.decision_function(
        X_val
    )


    # --------------------------------------------------------
    # Threshold from BENIGN validation only
    # --------------------------------------------------------

    benign_scores = scores[
        y_validation == 0
    ]


    threshold = np.percentile(

        benign_scores,

        THRESHOLD_PERCENTILE
    )


    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    predictions = (

        scores > threshold

    ).astype(
        np.int8
    )


    # --------------------------------------------------------
    # F1
    # --------------------------------------------------------

    f1 = f1_score(

        y_validation,

        predictions,

        zero_division=0
    )


    # --------------------------------------------------------
    # Feature penalty
    # --------------------------------------------------------

    feature_ratio = (

        chromosome.sum()
        /
        N_ES_FEATURES
    )


    # --------------------------------------------------------
    # Final fitness
    # --------------------------------------------------------

    fitness = (

        f1

        -

        FEATURE_PENALTY
        *
        feature_ratio
    )


    return fitness


# ============================================================
# 11. KNN FITNESS
# ============================================================

def es_knn_fitness(
    chromosome
):

    selected_columns = (
        es_chromosome_to_columns(
            chromosome
        )
    )


    # --------------------------------------------------------
    # Safety check
    # --------------------------------------------------------

    if len(selected_columns) == 0:

        return -1.0


    # --------------------------------------------------------
    # Select encoded columns
    # --------------------------------------------------------

    X_tr = X_train_scaled[
        :,
        selected_columns
    ]

    X_val = X_validation_scaled[
        :,
        selected_columns
    ]


    # --------------------------------------------------------
    # KNN
    # --------------------------------------------------------

    model = NearestNeighbors(

        n_neighbors=KNN_K,

        algorithm="auto",

        n_jobs=-1
    )


    model.fit(
        X_tr
    )


    # --------------------------------------------------------
    # Nearest-neighbour distances
    # --------------------------------------------------------

    distances, _ = model.kneighbors(

        X_val,

        n_neighbors=KNN_K
    )


    # --------------------------------------------------------
    # K-th nearest neighbour distance
    # --------------------------------------------------------

    scores = distances[:, -1]


    # --------------------------------------------------------
    # Threshold from BENIGN validation only
    # --------------------------------------------------------

    benign_scores = scores[
        y_validation == 0
    ]


    threshold = np.percentile(

        benign_scores,

        THRESHOLD_PERCENTILE
    )


    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    predictions = (

        scores > threshold

    ).astype(
        np.int8
    )


    # --------------------------------------------------------
    # F1
    # --------------------------------------------------------

    f1 = f1_score(

        y_validation,

        predictions,

        zero_division=0
    )


    # --------------------------------------------------------
    # Feature penalty
    # --------------------------------------------------------

    feature_ratio = (

        chromosome.sum()
        /
        N_ES_FEATURES
    )


    # --------------------------------------------------------
    # Final fitness
    # --------------------------------------------------------

    fitness = (

        f1

        -

        FEATURE_PENALTY
        *
        feature_ratio
    )


    return fitness


# ============================================================
# 12. GENERIC (1+1)-ES
# ============================================================

def run_one_plus_one_es(
    fitness_function,
    classifier_name
):

    print("\n")
    print("=" * 70)
    print(
        f"(1+1)-ES + {classifier_name}"
    )
    print("=" * 70)


    print(
        "\nIterations:",
        ES_ITERATIONS
    )

    print(
        "Mutation rate:",
        f"{ES_MUTATION_RATE:.4f}"
    )


    # ========================================================
    # START TIMER
    # ========================================================

    start_time = time.time()


    # ========================================================
    # INITIAL SOLUTION
    # ========================================================

    parent = (
        create_es_initial_chromosome()
    )


    parent_fitness = (
        fitness_function(
            parent
        )
    )


    # ========================================================
    # GLOBAL BEST
    # ========================================================

    best_chromosome = (
        parent.copy()
    )

    best_fitness = (
        parent_fitness
    )


    # ========================================================
    # HISTORY
    # ========================================================

    fitness_history = [
        best_fitness
    ]

    accepted_history = []

    feature_history = [
        int(parent.sum())
    ]


    # ========================================================
    # INITIAL RESULT
    # ========================================================

    print(
        "\nInitial solution:"
    )

    print(
        "  Chromosome:",
        parent
    )

    print(
        "  Features:",
        int(parent.sum()),
        "/",
        N_ES_FEATURES
    )

    print(
        "  Fitness:",
        f"{parent_fitness:.4f}"
    )


    # ========================================================
    # EVOLUTION
    # ========================================================

    for iteration in range(
        ES_ITERATIONS
    ):


        # ----------------------------------------------------
        # Generate ONE offspring
        # ----------------------------------------------------

        offspring = mutate_es(
            parent
        )


        # ----------------------------------------------------
        # Evaluate offspring
        # ----------------------------------------------------

        offspring_fitness = (
            fitness_function(
                offspring
            )
        )


        # ----------------------------------------------------
        # Selection
        #
        # (1+1)-ES:
        #
        # offspring replaces parent if it is at least
        # as good as parent.
        # ----------------------------------------------------

        if (
            offspring_fitness
            >=
            parent_fitness
        ):

            parent = (
                offspring.copy()
            )

            parent_fitness = (
                offspring_fitness
            )

            accepted = True

        else:

            accepted = False


        # ----------------------------------------------------
        # Update global best
        # ----------------------------------------------------

        if (
            parent_fitness
            >
            best_fitness
        ):

            best_fitness = (
                parent_fitness
            )

            best_chromosome = (
                parent.copy()
            )


        # ----------------------------------------------------
        # Save history
        # ----------------------------------------------------

        fitness_history.append(
            best_fitness
        )

        accepted_history.append(
            accepted
        )

        feature_history.append(
            int(parent.sum())
        )


        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        print(

            f"Iteration "
            f"{iteration + 1:>3}/"
            f"{ES_ITERATIONS}"

            f" | Parent fitness: "
            f"{parent_fitness:.4f}"

            f" | Best fitness: "
            f"{best_fitness:.4f}"

            f" | Features: "
            f"{parent.sum():>2}"

            f" | "
            f"{'ACCEPTED' if accepted else 'REJECTED'}"
        )


    # ========================================================
    # RUNTIME
    # ========================================================

    runtime = (
        time.time()
        -
        start_time
    )


    # ========================================================
    # SELECTED FEATURES
    # ========================================================

    selected_features = (
        es_chromosome_to_features(
            best_chromosome
        )
    )


    removed_features = [

        feature

        for feature in ES_FEATURES

        if feature
        not in selected_features
    ]


    selected_columns = (
        es_chromosome_to_columns(
            best_chromosome
        )
    )


    # ========================================================
    # FINAL OUTPUT
    # ========================================================

    print("\n")
    print("=" * 70)
    print(
        f"{classifier_name} "
        f"(1+1)-ES COMPLETE"
    )
    print("=" * 70)


    print(
        f"\nRuntime: "
        f"{runtime:.2f} seconds "
        f"({runtime / 60:.2f} minutes)"
    )


    print(
        f"Best fitness: "
        f"{best_fitness:.4f}"
    )


    print(
        f"Selected features: "
        f"{best_chromosome.sum()} / "
        f"{N_ES_FEATURES}"
    )


    print(
        "\nBest chromosome:"
    )

    print(
        best_chromosome
    )


    print(
        "\nSelected original features:"
    )

    for feature in selected_features:

        print(
            f"  + {feature}"
        )


    print(
        "\nRemoved original features:"
    )

    for feature in removed_features:

        print(
            f"  - {feature}"
        )


    print(
        "\nEncoded columns selected:",
        len(selected_columns)
    )


    acceptance_rate = (

        np.mean(
            accepted_history
        )

        if len(accepted_history) > 0

        else 0.0
    )


    print(
        f"Acceptance rate: "
        f"{acceptance_rate:.4f}"
    )


    # ========================================================
    # RETURN
    # ========================================================

    return {

        "classifier":
            classifier_name,

        "chromosome":
            best_chromosome,

        "fitness":
            best_fitness,

        "selected_features":
            selected_features,

        "removed_features":
            removed_features,

        "selected_columns":
            selected_columns,

        "fitness_history":
            fitness_history,

        "feature_history":
            feature_history,

        "accepted_history":
            accepted_history,

        "acceptance_rate":
            acceptance_rate,

        "runtime":
            runtime
    }


# ============================================================
# 13. RUN (1+1)-ES — SVM
# ============================================================

es_svm_results = run_one_plus_one_es(

    es_svm_fitness,

    "ONE-CLASS SVM"
)


# ============================================================
# 14. RUN (1+1)-ES — KNN
# ============================================================

es_knn_results = run_one_plus_one_es(

    es_knn_fitness,

    "KNN"
)


# ============================================================
# 15. PHASE D SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("PHASE D — (1+1)-ES SUMMARY")
print("=" * 70)


# ============================================================
# SVM
# ============================================================

print("\nONE-CLASS SVM")

print(
    f"Fitness: "
    f"{es_svm_results['fitness']:.4f}"
)

print(
    f"Features selected: "
    f"{len(es_svm_results['selected_features'])}"
)

print(
    "Selected:"
)

for feature in (
    es_svm_results[
        "selected_features"
    ]
):

    print(
        f"  + {feature}"
    )


# ============================================================
# KNN
# ============================================================

print("\nKNN")

print(
    f"Fitness: "
    f"{es_knn_results['fitness']:.4f}"
)

print(
    f"Features selected: "
    f"{len(es_knn_results['selected_features'])}"
)

print(
    "Selected:"
)

for feature in (
    es_knn_results[
        "selected_features"
    ]
):

    print(
        f"  + {feature}"
    )


# ============================================================
# PHASE D COMPLETE
# ============================================================

print("\n")
print("=" * 70)
print("PHASE D COMPLETE")
print("=" * 70)

print(
    "\nThe (1+1)-ES has finished feature selection."
)

print(
    "The final test set has NOT been used during evolution."
)

print(
    "\nNext:"
)

print(
    "  Evaluate the selected ES subsets on the"
)

print(
    "  untouched final test set."
)

PHASE D — (1+1)-EVOLUTIONARY STRATEGY

Original features:
  0: id.orig_p
  1: id.resp_p
  2: duration
  3: orig_bytes
  4: resp_bytes
  5: orig_pkts
  6: resp_pkts
  7: proto
  8: conn_state
  9: history

Total original features: 10
Mutation probability: 0.1000
Iterations: 50
Feature penalty: 0.01


ES FEATURE → ENCODED COLUMN MAPPING
id.orig_p       -> [0]
id.resp_p       -> [1]
duration        -> [2]
orig_bytes      -> [3]
resp_bytes      -> [4]
orig_pkts       -> [5]
resp_pkts       -> [6]
proto           -> [7, 8]
conn_state      -> [9, 10, 11]
history         -> [12, 13, 14, 15]


(1+1)-ES + ONE-CLASS SVM

Iterations: 50
Mutation rate: 0.1000

Initial solution:
  Chromosome: [1 1 1 1 1 1 1 1 1 1]
  Features: 10 / 10
  Fitness: 0.9850
Iteration   1/50 | Parent fitness: 0.9860 | Best fitness: 0.9860 | Features:  9 | ACCEPTED
Iteration   2/50 | Parent fitness: 0.9870 | Best fitness: 0.9870 | Features:  8 | ACCEPTED
Iteration   3/50 | Parent fitness: 0.9870 | Best fitness: 0.9870 | Fe

In [15]:
# ============================================================
# PHASE D.1 — (1+1)-ES FINAL TEST EVALUATION
# ============================================================
#
# Uses the BEST chromosomes found in Phase D.
#
# IMPORTANT:
#   - ES is already finished.
#   - No feature selection occurs here.
#   - Final test is not used to modify the solution.
#   - Threshold is determined using benign validation only.
#   - Final test is evaluated once.
#
# Methods:
#   1. (1+1)-ES + One-Class SVM
#   2. (1+1)-ES + KNN
# ============================================================


import time
import numpy as np

from sklearn.svm import OneClassSVM
from sklearn.neighbors import NearestNeighbors

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)


print("=" * 70)
print("PHASE D.1 — (1+1)-ES FINAL TEST EVALUATION")
print("=" * 70)


# ============================================================
# 1. GET BEST CHROMOSOMES
# ============================================================

es_svm_chromosome = (
    es_svm_results[
        "chromosome"
    ]
)

es_knn_chromosome = (
    es_knn_results[
        "chromosome"
    ]
)


# ============================================================
# 2. GET SELECTED COLUMNS
# ============================================================

es_svm_selected_columns = (
    es_chromosome_to_columns(
        es_svm_chromosome
    )
)

es_knn_selected_columns = (
    es_chromosome_to_columns(
        es_knn_chromosome
    )
)


# ============================================================
# 3. GET SELECTED FEATURES
# ============================================================

es_svm_selected_features = (
    es_chromosome_to_features(
        es_svm_chromosome
    )
)

es_knn_selected_features = (
    es_chromosome_to_features(
        es_knn_chromosome
    )
)


# ============================================================
# 4. PRINT SELECTED FEATURES
# ============================================================

print("\n")
print("=" * 70)
print("(1+1)-ES FEATURE SELECTION RESULTS")
print("=" * 70)


print("\nONE-CLASS SVM")

print(
    "Selected original features:"
)

for feature in es_svm_selected_features:

    print(
        f"  + {feature}"
    )

print(
    f"\nOriginal features: "
    f"{len(es_svm_selected_features)} / "
    f"{N_ES_FEATURES}"
)

print(
    f"Encoded columns: "
    f"{len(es_svm_selected_columns)}"
)


print("\nKNN")

print(
    "Selected original features:"
)

for feature in es_knn_selected_features:

    print(
        f"  + {feature}"
    )

print(
    f"\nOriginal features: "
    f"{len(es_knn_selected_features)} / "
    f"{N_ES_FEATURES}"
)

print(
    f"Encoded columns: "
    f"{len(es_knn_selected_columns)}"
)


# ============================================================
# 5. CREATE SELECTED DATASETS
# ============================================================

X_train_svm_es = (
    X_train_scaled[
        :,
        es_svm_selected_columns
    ]
)

X_validation_svm_es = (
    X_validation_scaled[
        :,
        es_svm_selected_columns
    ]
)

X_final_svm_es = (
    X_final_scaled[
        :,
        es_svm_selected_columns
    ]
)


X_train_knn_es = (
    X_train_scaled[
        :,
        es_knn_selected_columns
    ]
)

X_validation_knn_es = (
    X_validation_scaled[
        :,
        es_knn_selected_columns
    ]
)

X_final_knn_es = (
    X_final_scaled[
        :,
        es_knn_selected_columns
    ]
)


# ============================================================
# 6. ES + SVM
# ============================================================

print("\n")
print("=" * 70)
print("(1+1)-ES + ONE-CLASS SVM — FINAL TEST")
print("=" * 70)


print(
    "\nTraining shape:",
    X_train_svm_es.shape
)

print(
    "Validation shape:",
    X_validation_svm_es.shape
)

print(
    "Final test shape:",
    X_final_svm_es.shape
)


# ============================================================
# 7. TRAIN SVM
# ============================================================

es_svm_train_start = time.time()


es_svm_model = OneClassSVM(

    kernel=SVM_KERNEL,

    nu=SVM_NU
)


es_svm_model.fit(
    X_train_svm_es
)


es_svm_train_time = (
    time.time()
    -
    es_svm_train_start
)


print(
    f"\nTraining time: "
    f"{es_svm_train_time:.4f} seconds"
)


# ============================================================
# 8. VALIDATION SCORES
# ============================================================

es_svm_val_start = time.time()


es_svm_validation_scores = (
    es_svm_model.decision_function(
        X_validation_svm_es
    )
)


es_svm_validation_time = (
    time.time()
    -
    es_svm_val_start
)


# ============================================================
# 9. VALIDATION THRESHOLD
# ============================================================

es_svm_benign_scores = (
    es_svm_validation_scores[
        y_validation == 0
    ]
)


es_svm_threshold = np.percentile(

    es_svm_benign_scores,

    THRESHOLD_PERCENTILE
)


# ============================================================
# 10. VALIDATION PREDICTIONS
# ============================================================

es_svm_validation_predictions = (

    es_svm_validation_scores
    >
    es_svm_threshold

).astype(
    np.int8
)


# ============================================================
# 11. VALIDATION METRICS
# ============================================================

es_svm_val_accuracy = accuracy_score(

    y_validation,

    es_svm_validation_predictions
)


es_svm_val_precision = precision_score(

    y_validation,

    es_svm_validation_predictions,

    zero_division=0
)


es_svm_val_recall = recall_score(

    y_validation,

    es_svm_validation_predictions,

    zero_division=0
)


es_svm_val_f1 = f1_score(

    y_validation,

    es_svm_validation_predictions,

    zero_division=0
)


es_svm_val_auc = roc_auc_score(

    y_validation,

    es_svm_validation_scores
)


es_svm_val_cm = confusion_matrix(

    y_validation,

    es_svm_validation_predictions
)


# ============================================================
# 12. FINAL TEST SCORES
# ============================================================

es_svm_final_start = time.time()


es_svm_final_scores = (
    es_svm_model.decision_function(
        X_final_svm_es
    )
)


es_svm_final_prediction_time = (
    time.time()
    -
    es_svm_final_start
)


# ============================================================
# 13. FINAL TEST PREDICTIONS
# ============================================================

es_svm_final_predictions = (

    es_svm_final_scores
    >
    es_svm_threshold

).astype(
    np.int8
)


# ============================================================
# 14. FINAL TEST METRICS
# ============================================================

es_svm_final_accuracy = accuracy_score(

    y_final,

    es_svm_final_predictions
)


es_svm_final_precision = precision_score(

    y_final,

    es_svm_final_predictions,

    zero_division=0
)


es_svm_final_recall = recall_score(

    y_final,

    es_svm_final_predictions,

    zero_division=0
)


es_svm_final_f1 = f1_score(

    y_final,

    es_svm_final_predictions,

    zero_division=0
)


es_svm_final_auc = roc_auc_score(

    y_final,

    es_svm_final_scores
)


es_svm_final_cm = confusion_matrix(

    y_final,

    es_svm_final_predictions
)


# ============================================================
# 15. PRINT SVM RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("(1+1)-ES + ONE-CLASS SVM — VALIDATION")
print("=" * 70)


print(
    f"Threshold : "
    f"{es_svm_threshold:.10f}"
)

print(
    f"Accuracy  : "
    f"{es_svm_val_accuracy:.4f}"
)

print(
    f"Precision : "
    f"{es_svm_val_precision:.4f}"
)

print(
    f"Recall    : "
    f"{es_svm_val_recall:.4f}"
)

print(
    f"F1        : "
    f"{es_svm_val_f1:.4f}"
)

print(
    f"ROC-AUC   : "
    f"{es_svm_val_auc:.4f}"
)

print(
    "\nConfusion matrix:"
)

print(
    es_svm_val_cm
)


print("\n")
print("=" * 70)
print("(1+1)-ES + ONE-CLASS SVM — FINAL TEST")
print("=" * 70)


print(
    f"Accuracy  : "
    f"{es_svm_final_accuracy:.4f}"
)

print(
    f"Precision : "
    f"{es_svm_final_precision:.4f}"
)

print(
    f"Recall    : "
    f"{es_svm_final_recall:.4f}"
)

print(
    f"F1        : "
    f"{es_svm_final_f1:.4f}"
)

print(
    f"ROC-AUC   : "
    f"{es_svm_final_auc:.4f}"
)

print(
    "\nConfusion matrix:"
)

print(
    es_svm_final_cm
)

print(
    f"\nFinal prediction time: "
    f"{es_svm_final_prediction_time:.4f} seconds"
)


# ============================================================
# 16. ES + KNN
# ============================================================

print("\n")
print("=" * 70)
print("(1+1)-ES + KNN — FINAL TEST")
print("=" * 70)


print(
    "\nTraining shape:",
    X_train_knn_es.shape
)

print(
    "Validation shape:",
    X_validation_knn_es.shape
)

print(
    "Final test shape:",
    X_final_knn_es.shape
)


# ============================================================
# 17. TRAIN KNN
# ============================================================

es_knn_train_start = time.time()


es_knn_model = NearestNeighbors(

    n_neighbors=KNN_K,

    algorithm="auto",

    n_jobs=-1
)


es_knn_model.fit(
    X_train_knn_es
)


es_knn_train_time = (
    time.time()
    -
    es_knn_train_start
)


print(
    f"\nTraining time: "
    f"{es_knn_train_time:.4f} seconds"
)


# ============================================================
# 18. VALIDATION SCORES
# ============================================================

es_knn_val_start = time.time()


es_knn_validation_distances, _ = (
    es_knn_model.kneighbors(

        X_validation_knn_es,

        n_neighbors=KNN_K
    )
)


es_knn_validation_scores = (
    es_knn_validation_distances[
        :,
        -1
    ]
)


es_knn_validation_time = (
    time.time()
    -
    es_knn_val_start
)


# ============================================================
# 19. VALIDATION THRESHOLD
# ============================================================

es_knn_benign_scores = (
    es_knn_validation_scores[
        y_validation == 0
    ]
)


es_knn_threshold = np.percentile(

    es_knn_benign_scores,

    THRESHOLD_PERCENTILE
)


# ============================================================
# 20. VALIDATION PREDICTIONS
# ============================================================

es_knn_validation_predictions = (

    es_knn_validation_scores
    >
    es_knn_threshold

).astype(
    np.int8
)


# ============================================================
# 21. VALIDATION METRICS
# ============================================================

es_knn_val_accuracy = accuracy_score(

    y_validation,

    es_knn_validation_predictions
)


es_knn_val_precision = precision_score(

    y_validation,

    es_knn_validation_predictions,

    zero_division=0
)


es_knn_val_recall = recall_score(

    y_validation,

    es_knn_validation_predictions,

    zero_division=0
)


es_knn_val_f1 = f1_score(

    y_validation,

    es_knn_validation_predictions,

    zero_division=0
)


es_knn_val_auc = roc_auc_score(

    y_validation,

    es_knn_validation_scores
)


es_knn_val_cm = confusion_matrix(

    y_validation,

    es_knn_validation_predictions
)


# ============================================================
# 22. FINAL TEST SCORES
# ============================================================

es_knn_final_start = time.time()


es_knn_final_distances, _ = (
    es_knn_model.kneighbors(

        X_final_knn_es,

        n_neighbors=KNN_K
    )
)


es_knn_final_scores = (
    es_knn_final_distances[
        :,
        -1
    ]
)


es_knn_final_prediction_time = (
    time.time()
    -
    es_knn_final_start
)


# ============================================================
# 23. FINAL TEST PREDICTIONS
# ============================================================

es_knn_final_predictions = (

    es_knn_final_scores
    >
    es_knn_threshold

).astype(
    np.int8
)


# ============================================================
# 24. FINAL TEST METRICS
# ============================================================

es_knn_final_accuracy = accuracy_score(

    y_final,

    es_knn_final_predictions
)


es_knn_final_precision = precision_score(

    y_final,

    es_knn_final_predictions,

    zero_division=0
)


es_knn_final_recall = recall_score(

    y_final,

    es_knn_final_predictions,

    zero_division=0
)


es_knn_final_f1 = f1_score(

    y_final,

    es_knn_final_predictions,

    zero_division=0
)


es_knn_final_auc = roc_auc_score(

    y_final,

    es_knn_final_scores
)


es_knn_final_cm = confusion_matrix(

    y_final,

    es_knn_final_predictions
)


# ============================================================
# 25. PRINT KNN RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("(1+1)-ES + KNN — VALIDATION")
print("=" * 70)


print(
    f"Threshold : "
    f"{es_knn_threshold:.10f}"
)

print(
    f"Accuracy  : "
    f"{es_knn_val_accuracy:.4f}"
)

print(
    f"Precision : "
    f"{es_knn_val_precision:.4f}"
)

print(
    f"Recall    : "
    f"{es_knn_val_recall:.4f}"
)

print(
    f"F1        : "
    f"{es_knn_val_f1:.4f}"
)

print(
    f"ROC-AUC   : "
    f"{es_knn_val_auc:.4f}"
)

print(
    "\nConfusion matrix:"
)

print(
    es_knn_val_cm
)


print("\n")
print("=" * 70)
print("(1+1)-ES + KNN — FINAL TEST")
print("=" * 70)


print(
    f"Accuracy  : "
    f"{es_knn_final_accuracy:.4f}"
)

print(
    f"Precision : "
    f"{es_knn_final_precision:.4f}"
)

print(
    f"Recall    : "
    f"{es_knn_final_recall:.4f}"
)

print(
    f"F1        : "
    f"{es_knn_final_f1:.4f}"
)

print(
    f"ROC-AUC   : "
    f"{es_knn_final_auc:.4f}"
)

print(
    "\nConfusion matrix:"
)

print(
    es_knn_final_cm
)

print(
    f"\nFinal prediction time: "
    f"{es_knn_final_prediction_time:.4f} seconds"
)


# ============================================================
# 26. FEATURE REDUCTION
# ============================================================

es_svm_reduction = (

    1
    -
    len(es_svm_selected_features)
    /
    N_ES_FEATURES

) * 100


es_knn_reduction = (

    1
    -
    len(es_knn_selected_features)
    /
    N_ES_FEATURES

) * 100


print("\n")
print("=" * 70)
print("PHASE D — FEATURE REDUCTION")
print("=" * 70)


print("\nONE-CLASS SVM")

print(
    f"  {N_ES_FEATURES} → "
    f"{len(es_svm_selected_features)} features"
)

print(
    f"  Reduction: "
    f"{es_svm_reduction:.1f}%"
)


print("\nKNN")

print(
    f"  {N_ES_FEATURES} → "
    f"{len(es_knn_selected_features)} features"
)

print(
    f"  Reduction: "
    f"{es_knn_reduction:.1f}%"
)


# ============================================================
# 27. STORE RESULTS
# ============================================================

phase_d_final_results = {

    "svm": {

        "selected_features":
            es_svm_selected_features,

        "selected_columns":
            es_svm_selected_columns,

        "n_features":
            len(es_svm_selected_features),

        "fitness":
            es_svm_results["fitness"],

        "accuracy":
            es_svm_final_accuracy,

        "precision":
            es_svm_final_precision,

        "recall":
            es_svm_final_recall,

        "f1":
            es_svm_final_f1,

        "auc":
            es_svm_final_auc,

        "confusion_matrix":
            es_svm_final_cm,

        "train_time":
            es_svm_train_time,

        "prediction_time":
            es_svm_final_prediction_time,

        "threshold":
            es_svm_threshold
    },


    "knn": {

        "selected_features":
            es_knn_selected_features,

        "selected_columns":
            es_knn_selected_columns,

        "n_features":
            len(es_knn_selected_features),

        "fitness":
            es_knn_results["fitness"],

        "accuracy":
            es_knn_final_accuracy,

        "precision":
            es_knn_final_precision,

        "recall":
            es_knn_final_recall,

        "f1":
            es_knn_final_f1,

        "auc":
            es_knn_final_auc,

        "confusion_matrix":
            es_knn_final_cm,

        "train_time":
            es_knn_train_time,

        "prediction_time":
            es_knn_final_prediction_time,

        "threshold":
            es_knn_threshold
    }
}


# ============================================================
# 28. FINAL PHASE D COMPARISON
# ============================================================

print("\n")
print("=" * 70)
print("PHASE D — FULL FEATURES vs GA vs (1+1)-ES")
print("=" * 70)


print(
    f"\n{'Metric':<30}"
    f"{'SVM Full':>12}"
    f"{'SVM GA':>12}"
    f"{'SVM ES':>12}"
    f"{'KNN Full':>12}"
    f"{'KNN GA':>12}"
    f"{'KNN ES':>12}"
)

print("-" * 102)


print(
    f"{'Original features':<30}"
    f"{10:>12}"
    f"{len(svm_selected_features):>12}"
    f"{len(es_svm_selected_features):>12}"
    f"{10:>12}"
    f"{len(knn_selected_features):>12}"
    f"{len(es_knn_selected_features):>12}"
)


print(
    f"{'Encoded features':<30}"
    f"{X_train.shape[1]:>12}"
    f"{len(svm_selected_columns):>12}"
    f"{len(es_svm_selected_columns):>12}"
    f"{X_train.shape[1]:>12}"
    f"{len(knn_selected_columns):>12}"
    f"{len(es_knn_selected_columns):>12}"
)


print(
    f"{'Final Accuracy':<30}"
    f"{svm_results['final_accuracy']:>12.4f}"
    f"{svm_ga_final_accuracy:>12.4f}"
    f"{es_svm_final_accuracy:>12.4f}"
    f"{knn_results['final_accuracy']:>12.4f}"
    f"{knn_ga_final_accuracy:>12.4f}"
    f"{es_knn_final_accuracy:>12.4f}"
)


print(
    f"{'Final Precision':<30}"
    f"{svm_results['final_precision']:>12.4f}"
    f"{svm_ga_final_precision:>12.4f}"
    f"{es_svm_final_precision:>12.4f}"
    f"{knn_results['final_precision']:>12.4f}"
    f"{knn_ga_final_precision:>12.4f}"
    f"{es_knn_final_precision:>12.4f}"
)


print(
    f"{'Final Recall':<30}"
    f"{svm_results['final_recall']:>12.4f}"
    f"{svm_ga_final_recall:>12.4f}"
    f"{es_svm_final_recall:>12.4f}"
    f"{knn_results['final_recall']:>12.4f}"
    f"{knn_ga_final_recall:>12.4f}"
    f"{es_knn_final_recall:>12.4f}"
)


print(
    f"{'Final F1':<30}"
    f"{svm_results['final_f1']:>12.4f}"
    f"{svm_ga_final_f1:>12.4f}"
    f"{es_svm_final_f1:>12.4f}"
    f"{knn_results['final_f1']:>12.4f}"
    f"{knn_ga_final_f1:>12.4f}"
    f"{es_knn_final_f1:>12.4f}"
)


print(
    f"{'Final ROC-AUC':<30}"
    f"{svm_results['final_auc']:>12.4f}"
    f"{svm_ga_final_auc:>12.4f}"
    f"{es_svm_final_auc:>12.4f}"
    f"{knn_results['final_auc']:>12.4f}"
    f"{knn_ga_final_auc:>12.4f}"
    f"{es_knn_final_auc:>12.4f}"
)


print("\n")
print("=" * 70)
print("PHASE D.1 COMPLETE")
print("=" * 70)

print(
    "\n(1+1)-ES selected feature subsets have been"
)

print(
    "evaluated on the untouched 100,000-row final test set."
)

print(
    "\nNext phase:"
)

print(
    "  Phase E — Binary Chaotic Genetic Algorithm"
)

PHASE D.1 — (1+1)-ES FINAL TEST EVALUATION


(1+1)-ES FEATURE SELECTION RESULTS

ONE-CLASS SVM
Selected original features:
  + id.resp_p

Original features: 1 / 10
Encoded columns: 1

KNN
Selected original features:
  + id.resp_p
  + orig_pkts

Original features: 2 / 10
Encoded columns: 2


(1+1)-ES + ONE-CLASS SVM — FINAL TEST

Training shape: (10000, 1)
Validation shape: (5000, 1)
Final test shape: (100000, 1)

Training time: 0.0430 seconds


(1+1)-ES + ONE-CLASS SVM — VALIDATION
Threshold : 0.0016651074
Accuracy  : 1.0000
Precision : 1.0000
Recall    : 1.0000
F1        : 1.0000
ROC-AUC   : 1.0000

Confusion matrix:
[[2500    0]
 [   0 2500]]


(1+1)-ES + ONE-CLASS SVM — FINAL TEST
Accuracy  : 1.0000
Precision : 0.9999
Recall    : 1.0000
F1        : 1.0000
ROC-AUC   : 1.0000

Confusion matrix:
[[49997     3]
 [    0 50000]]

Final prediction time: 0.3725 seconds


(1+1)-ES + KNN — FINAL TEST

Training shape: (10000, 2)
Validation shape: (5000, 2)
Final test shape: (100000, 2)

Traini

In [16]:
# ================================================================
# PHASE E — BINARY CHAOTIC GENETIC ALGORITHM
# ================================================================

import numpy as np
import pandas as pd
import time
import warnings

from sklearn.svm import OneClassSVM
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")


print("=" * 70)
print("PHASE E — BINARY CHAOTIC GENETIC ALGORITHM")
print("=" * 70)


# ================================================================
# 1. CONFIGURATION
# ================================================================

BCGA_CONFIG = {

    # Population
    "population_size": 30,

    # Evolution
    "generations": 30,

    # Genetic operators
    "crossover_rate": 0.80,
    "mutation_rate": 0.05,

    # Elitism
    "elite_count": 2,

    # Feature-selection penalty
    "feature_penalty": 0.01,

    # Random seed
    "random_state": 42,

    # Chaotic map parameter
    "chaos_mu": 4.0,

    # Minimum number of features
    "min_features": 1,

    # Maximum number of features
    "max_features": None,
}


# ================================================================
# 2. FEATURE LIST
# ================================================================

FEATURE_COLUMNS = [
    "id.resp_p",
    "orig_pkts",
    "orig_ip_bytes",
    "resp_pkts",
    "resp_ip_bytes",
    "duration",
    "orig_bytes",
    "resp_bytes",
    "missed_bytes",
    "orig_bytes"
]

# Remove accidental duplicates while preserving order
FEATURE_COLUMNS = list(dict.fromkeys(FEATURE_COLUMNS))

print("\nCandidate features:")
for i, feature in enumerate(FEATURE_COLUMNS):
    print(f"  {i:2d}: {feature}")

N_FEATURES = len(FEATURE_COLUMNS)

print(f"\nTotal candidate features: {N_FEATURES}")


# ================================================================
# 3. VERIFY FEATURES EXIST
# ================================================================

missing_features = [
    f for f in FEATURE_COLUMNS
    if f not in train_df.columns
]

if missing_features:
    raise ValueError(
        f"Missing features from train_df: {missing_features}"
    )


# ================================================================
# 4. PREPARE DATA
# ================================================================

X_train_all = train_df[FEATURE_COLUMNS].copy()
X_val_all = val_df[FEATURE_COLUMNS].copy()
X_test_all = test_df[FEATURE_COLUMNS].copy()

y_train = train_df["label"].values
y_val = val_df["label"].values
y_test = test_df["label"].values


# ================================================================
# 5. SAFE NUMERIC CONVERSION
# ================================================================

def prepare_numeric(df):
    """
    Convert selected feature dataframe to numeric.
    """

    result = df.copy()

    for col in result.columns:
        result[col] = pd.to_numeric(
            result[col],
            errors="coerce"
        )

    result = result.replace(
        [np.inf, -np.inf],
        np.nan
    )

    result = result.fillna(0)

    return result


X_train_all = prepare_numeric(X_train_all)
X_val_all = prepare_numeric(X_val_all)
X_test_all = prepare_numeric(X_test_all)


# ================================================================
# 6. CHAOTIC MAP
# ================================================================

class ChaoticSequence:

    def __init__(self, seed=0.731, mu=4.0):

        self.x = seed
        self.mu = mu

    def next(self):

        self.x = self.mu * self.x * (1.0 - self.x)

        # Avoid exactly 0 or 1
        self.x = np.clip(
            self.x,
            1e-10,
            1 - 1e-10
        )

        return self.x


# ================================================================
# 7. BINARY CHAOTIC INITIALIZATION
# ================================================================

def initialize_population(
    population_size,
    n_features,
    chaotic_generator,
    min_features=1,
    max_features=None
):

    population = []

    if max_features is None:
        max_features = n_features

    for _ in range(population_size):

        chromosome = np.array([
            1 if chaotic_generator.next() >= 0.5 else 0
            for _ in range(n_features)
        ])

        # Force minimum number of selected features
        if chromosome.sum() < min_features:

            selected = int(
                chaotic_generator.next() * n_features
            )

            chromosome[selected] = 1

        # Enforce maximum number of features
        while chromosome.sum() > max_features:

            selected_indices = np.where(
                chromosome == 1
            )[0]

            selected = selected_indices[
                int(
                    chaotic_generator.next()
                    * len(selected_indices)
                )
            ]

            chromosome[selected] = 0

        population.append(chromosome)

    return np.array(population)


# ================================================================
# 8. ONE-CLASS SVM FITNESS
# ================================================================

def evaluate_svm_chromosome(
    chromosome,
    X_train,
    y_train,
    feature_columns,
    feature_penalty
):

    selected_indices = np.where(chromosome == 1)[0]

    if len(selected_indices) == 0:
        return -np.inf

    selected_features = [
        feature_columns[i]
        for i in selected_indices
    ]

    X = X_train[selected_features].values

    # Train only on benign samples
    X_benign = X[y_train == 0]

    if len(X_benign) == 0:
        return -np.inf

    model = OneClassSVM(
        kernel="rbf",
        gamma="scale",
        nu=0.05
    )

    try:

        model.fit(X_benign)

        predictions = model.predict(X)

        predictions = np.where(
            predictions == 1,
            0,
            1
        )

        f1 = f1_score(
            y_train,
            predictions,
            zero_division=0
        )

        reduction = (
            1 -
            len(selected_indices) /
            len(feature_columns)
        )

        fitness = (
            f1 +
            feature_penalty * reduction
        )

        return fitness

    except Exception:

        return -np.inf


# ================================================================
# 9. KNN FITNESS
# ================================================================

def evaluate_knn_chromosome(
    chromosome,
    X_train,
    y_train,
    feature_columns,
    feature_penalty
):

    selected_indices = np.where(chromosome == 1)[0]

    if len(selected_indices) == 0:
        return -np.inf

    selected_features = [
        feature_columns[i]
        for i in selected_indices
    ]

    X = X_train[selected_features].values

    try:

        # Nearest-neighbour model
        nn = NearestNeighbors(
            n_neighbors=5,
            metric="euclidean"
        )

        nn.fit(X[y_train == 0])

        distances, _ = nn.kneighbors(X)

        anomaly_scores = distances.mean(axis=1)

        # Training threshold
        threshold = np.percentile(
            anomaly_scores[y_train == 0],
            95
        )

        predictions = (
            anomaly_scores > threshold
        ).astype(int)

        f1 = f1_score(
            y_train,
            predictions,
            zero_division=0
        )

        reduction = (
            1 -
            len(selected_indices) /
            len(feature_columns)
        )

        fitness = (
            f1 +
            feature_penalty * reduction
        )

        return fitness

    except Exception:

        return -np.inf


# ================================================================
# 10. CHAOTIC SELECTION
# ================================================================

def chaotic_tournament_selection(
    population,
    fitness,
    chaotic_generator,
    tournament_size=3
):

    candidates = []

    for _ in range(tournament_size):

        index = int(
            chaotic_generator.next()
            * len(population)
        )

        candidates.append(index)

    best_index = max(
        candidates,
        key=lambda i: fitness[i]
    )

    return population[best_index].copy()


# ================================================================
# 11. CHAOTIC SINGLE-POINT CROSSOVER
# ================================================================

def chaotic_crossover(
    parent1,
    parent2,
    chaotic_generator,
    crossover_rate
):

    if chaotic_generator.next() > crossover_rate:

        return parent1.copy(), parent2.copy()

    point = int(
        chaotic_generator.next()
        * (len(parent1) - 1)
    ) + 1

    child1 = np.concatenate([
        parent1[:point],
        parent2[point:]
    ])

    child2 = np.concatenate([
        parent2[:point],
        parent1[point:]
    ])

    return child1, child2


# ================================================================
# 12. CHAOTIC BIT MUTATION
# ================================================================

def chaotic_mutation(
    chromosome,
    chaotic_generator,
    mutation_rate,
    min_features,
    max_features
):

    mutated = chromosome.copy()

    for i in range(len(mutated)):

        if chaotic_generator.next() < mutation_rate:

            mutated[i] = 1 - mutated[i]

    # Minimum features
    if mutated.sum() < min_features:

        index = int(
            chaotic_generator.next()
            * len(mutated)
        )

        mutated[index] = 1

    # Maximum features
    while mutated.sum() > max_features:

        selected_indices = np.where(
            mutated == 1
        )[0]

        index = selected_indices[
            int(
                chaotic_generator.next()
                * len(selected_indices)
            )
        ]

        mutated[index] = 0

    return mutated


# ================================================================
# 13. BCGA OPTIMIZER
# ================================================================

def run_bcga(
    model_type,
    X_train,
    y_train,
    feature_columns,
    config
):

    print("\n" + "=" * 70)
    print(f"BCGA FEATURE SELECTION — {model_type}")
    print("=" * 70)

    rng = np.random.default_rng(
        config["random_state"]
    )

    chaotic = ChaoticSequence(
        seed=0.731,
        mu=config["chaos_mu"]
    )

    population = initialize_population(
        population_size=config["population_size"],
        n_features=len(feature_columns),
        chaotic_generator=chaotic,
        min_features=config["min_features"],
        max_features=(
            config["max_features"]
            if config["max_features"] is not None
            else len(feature_columns)
        )
    )

    best_chromosome = None
    best_fitness = -np.inf

    history = []

    start_time = time.time()

    for generation in range(
        config["generations"]
    ):

        # --------------------------------------------------------
        # FITNESS
        # --------------------------------------------------------

        fitness_values = []

        for chromosome in population:

            if model_type == "SVM":

                fitness = evaluate_svm_chromosome(
                    chromosome,
                    X_train,
                    y_train,
                    feature_columns,
                    config["feature_penalty"]
                )

            elif model_type == "KNN":

                fitness = evaluate_knn_chromosome(
                    chromosome,
                    X_train,
                    y_train,
                    feature_columns,
                    config["feature_penalty"]
                )

            else:

                raise ValueError(
                    "Unknown model type"
                )

            fitness_values.append(fitness)

        fitness_values = np.array(
            fitness_values
        )

        # --------------------------------------------------------
        # BEST
        # --------------------------------------------------------

        generation_best = np.argmax(
            fitness_values
        )

        generation_best_fitness = (
            fitness_values[generation_best]
        )

        if generation_best_fitness > best_fitness:

            best_fitness = generation_best_fitness

            best_chromosome = population[
                generation_best
            ].copy()

        history.append({
            "generation": generation,
            "best_fitness": best_fitness,
            "selected_features": int(
                best_chromosome.sum()
            )
        })

        print(
            f"Generation {generation + 1:3d}/{config['generations']} "
            f"| Fitness: {best_fitness:.6f} "
            f"| Features: {best_chromosome.sum()}"
        )

        # --------------------------------------------------------
        # SORT POPULATION
        # --------------------------------------------------------

        order = np.argsort(
            fitness_values
        )[::-1]

        population = population[order]

        # --------------------------------------------------------
        # ELITISM
        # --------------------------------------------------------

        new_population = [
            population[i].copy()
            for i in range(
                config["elite_count"]
            )
        ]

        # --------------------------------------------------------
        # REPRODUCTION
        # --------------------------------------------------------

        while len(new_population) < config[
            "population_size"
        ]:

            parent1 = chaotic_tournament_selection(
                population,
                fitness_values[order],
                chaotic,
                tournament_size=3
            )

            parent2 = chaotic_tournament_selection(
                population,
                fitness_values[order],
                chaotic,
                tournament_size=3
            )

            child1, child2 = chaotic_crossover(
                parent1,
                parent2,
                chaotic,
                config["crossover_rate"]
            )

            child1 = chaotic_mutation(
                child1,
                chaotic,
                config["mutation_rate"],
                config["min_features"],
                (
                    config["max_features"]
                    if config["max_features"] is not None
                    else len(feature_columns)
                )
            )

            child2 = chaotic_mutation(
                child2,
                chaotic,
                config["mutation_rate"],
                config["min_features"],
                (
                    config["max_features"]
                    if config["max_features"] is not None
                    else len(feature_columns)
                )
            )

            new_population.append(child1)

            if len(new_population) < config[
                "population_size"
            ]:

                new_population.append(child2)

        population = np.array(
            new_population
        )

    elapsed = time.time() - start_time

    selected_indices = np.where(
        best_chromosome == 1
    )[0]

    selected_features = [
        feature_columns[i]
        for i in selected_indices
    ]

    print("\n" + "-" * 70)
    print(f"BCGA COMPLETE — {model_type}")
    print("-" * 70)

    print(
        f"Best fitness: {best_fitness:.6f}"
    )

    print(
        f"Selected features: "
        f"{len(selected_features)} / "
        f"{len(feature_columns)}"
    )

    for feature in selected_features:

        print(f"  + {feature}")

    print(
        f"Runtime: {elapsed:.3f} seconds"
    )

    return {
        "model": model_type,
        "chromosome": best_chromosome,
        "selected_features": selected_features,
        "fitness": best_fitness,
        "history": history,
        "runtime": elapsed
    }


# ================================================================
# 14. RUN BCGA — SVM
# ================================================================

bcga_svm_result = run_bcga(
    model_type="SVM",
    X_train=X_train_all,
    y_train=y_train,
    feature_columns=FEATURE_COLUMNS,
    config=BCGA_CONFIG
)


# ================================================================
# 15. RUN BCGA — KNN
# ================================================================

bcga_knn_result = run_bcga(
    model_type="KNN",
    X_train=X_train_all,
    y_train=y_train,
    feature_columns=FEATURE_COLUMNS,
    config=BCGA_CONFIG
)


# ================================================================
# 16. DISPLAY FEATURE SELECTION RESULTS
# ================================================================

print("\n")
print("=" * 70)
print("BCGA FEATURE SELECTION RESULTS")
print("=" * 70)

for result in [
    bcga_svm_result,
    bcga_knn_result
]:

    print(f"\n{result['model']}")

    print("Selected original features:")

    for feature in result["selected_features"]:

        print(f"  + {feature}")

    print(
        f"\nOriginal features: "
        f"{len(result['selected_features'])} / "
        f"{len(FEATURE_COLUMNS)}"
    )

    print(
        f"Encoded columns: "
        f"{len(result['selected_features'])}"
    )


# ================================================================
# PHASE E.1 — BCGA FINAL TEST EVALUATION
# ================================================================

print("\n\n")
print("=" * 70)
print("PHASE E.1 — BCGA FINAL TEST EVALUATION")
print("=" * 70)


# ================================================================
# 17. SVM EVALUATION
# ================================================================

def evaluate_bcga_svm(
    selected_features,
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test
):

    print("\n")
    print("=" * 70)
    print("BCGA + ONE-CLASS SVM — FINAL TEST")
    print("=" * 70)

    Xtr = X_train[selected_features].values
    Xv = X_val[selected_features].values
    Xt = X_test[selected_features].values

    # ------------------------------------------------------------
    # Train only on benign
    # ------------------------------------------------------------

    Xtr_benign = Xtr[y_train == 0]

    print(
        f"\nTraining shape: {Xtr.shape}"
    )

    print(
        f"Validation shape: {Xv.shape}"
    )

    print(
        f"Final test shape: {Xt.shape}"
    )

    start = time.time()

    model = OneClassSVM(
        kernel="rbf",
        gamma="scale",
        nu=0.05
    )

    model.fit(Xtr_benign)

    training_time = time.time() - start

    print(
        f"\nTraining time: "
        f"{training_time:.4f} seconds"
    )

    # ------------------------------------------------------------
    # Validation scores
    # ------------------------------------------------------------

    val_scores = -model.decision_function(Xv)

    benign_val_scores = val_scores[
        y_val == 0
    ]

    # Percentile threshold
    threshold = np.percentile(
        benign_val_scores,
        99
    )

    val_pred = (
        val_scores >= threshold
    ).astype(int)

    # ------------------------------------------------------------
    # Validation metrics
    # ------------------------------------------------------------

    print("\n")
    print("=" * 70)
    print("BCGA + ONE-CLASS SVM — VALIDATION")
    print("=" * 70)

    print(
        f"Threshold : {threshold:.10f}"
    )

    print(
        f"Accuracy  : "
        f"{accuracy_score(y_val, val_pred):.4f}"
    )

    print(
        f"Precision : "
        f"{precision_score(y_val, val_pred, zero_division=0):.4f}"
    )

    print(
        f"Recall    : "
        f"{recall_score(y_val, val_pred, zero_division=0):.4f}"
    )

    print(
        f"F1        : "
        f"{f1_score(y_val, val_pred, zero_division=0):.4f}"
    )

    print(
        f"ROC-AUC   : "
        f"{roc_auc_score(y_val, val_scores):.4f}"
    )

    print("\nConfusion matrix:")

    print(
        confusion_matrix(
            y_val,
            val_pred
        )
    )

    # ------------------------------------------------------------
    # FINAL TEST
    # ------------------------------------------------------------

    start = time.time()

    test_scores = -model.decision_function(Xt)

    test_pred = (
        test_scores >= threshold
    ).astype(int)

    prediction_time = time.time() - start

    print("\n")
    print("=" * 70)
    print("BCGA + ONE-CLASS SVM — FINAL TEST")
    print("=" * 70)

    print(
        f"Accuracy  : "
        f"{accuracy_score(y_test, test_pred):.4f}"
    )

    print(
        f"Precision : "
        f"{precision_score(y_test, test_pred, zero_division=0):.4f}"
    )

    print(
        f"Recall    : "
        f"{recall_score(y_test, test_pred, zero_division=0):.4f}"
    )

    print(
        f"F1        : "
        f"{f1_score(y_test, test_pred, zero_division=0):.4f}"
    )

    print(
        f"ROC-AUC   : "
        f"{roc_auc_score(y_test, test_scores):.4f}"
    )

    print("\nConfusion matrix:")

    print(
        confusion_matrix(
            y_test,
            test_pred
        )
    )

    print(
        f"\nFinal prediction time: "
        f"{prediction_time:.4f} seconds"
    )

    return {
        "model": model,
        "threshold": threshold,
        "val_scores": val_scores,
        "test_scores": test_scores,
        "val_predictions": val_pred,
        "test_predictions": test_pred,
        "training_time": training_time,
        "prediction_time": prediction_time
    }


# ================================================================
# 18. KNN EVALUATION
# ================================================================

def evaluate_bcga_knn(
    selected_features,
    X_train,
    y_train,
    X_val,
    y_val,
    X_test,
    y_test
):

    print("\n")
    print("=" * 70)
    print("BCGA + KNN — FINAL TEST")
    print("=" * 70)

    Xtr = X_train[selected_features].values
    Xv = X_val[selected_features].values
    Xt = X_test[selected_features].values

    print(
        f"\nTraining shape: {Xtr.shape}"
    )

    print(
        f"Validation shape: {Xv.shape}"
    )

    print(
        f"Final test shape: {Xt.shape}"
    )

    # ------------------------------------------------------------
    # Train
    # ------------------------------------------------------------

    start = time.time()

    Xtr_benign = Xtr[
        y_train == 0
    ]

    nn = NearestNeighbors(
        n_neighbors=5,
        metric="euclidean"
    )

    nn.fit(Xtr_benign)

    training_time = time.time() - start

    print(
        f"\nTraining time: "
        f"{training_time:.4f} seconds"
    )

    # ------------------------------------------------------------
    # Validation
    # ------------------------------------------------------------

    val_distances, _ = nn.kneighbors(Xv)

    val_scores = val_distances.mean(
        axis=1
    )

    benign_val_scores = val_scores[
        y_val == 0
    ]

    threshold = np.percentile(
        benign_val_scores,
        99
    )

    val_pred = (
        val_scores >= threshold
    ).astype(int)

    # ------------------------------------------------------------
    # Validation metrics
    # ------------------------------------------------------------

    print("\n")
    print("=" * 70)
    print("BCGA + KNN — VALIDATION")
    print("=" * 70)

    print(
        f"Threshold : {threshold:.10f}"
    )

    print(
        f"Accuracy  : "
        f"{accuracy_score(y_val, val_pred):.4f}"
    )

    print(
        f"Precision : "
        f"{precision_score(y_val, val_pred, zero_division=0):.4f}"
    )

    print(
        f"Recall    : "
        f"{recall_score(y_val, val_pred, zero_division=0):.4f}"
    )

    print(
        f"F1        : "
        f"{f1_score(y_val, val_pred, zero_division=0):.4f}"
    )

    print(
        f"ROC-AUC   : "
        f"{roc_auc_score(y_val, val_scores):.4f}"
    )

    print("\nConfusion matrix:")

    print(
        confusion_matrix(
            y_val,
            val_pred
        )
    )

    # ------------------------------------------------------------
    # FINAL TEST
    # ------------------------------------------------------------

    start = time.time()

    test_distances, _ = nn.kneighbors(Xt)

    test_scores = test_distances.mean(
        axis=1
    )

    test_pred = (
        test_scores >= threshold
    ).astype(int)

    prediction_time = time.time() - start

    print("\n")
    print("=" * 70)
    print("BCGA + KNN — FINAL TEST")
    print("=" * 70)

    print(
        f"Accuracy  : "
        f"{accuracy_score(y_test, test_pred):.4f}"
    )

    print(
        f"Precision : "
        f"{precision_score(y_test, test_pred, zero_division=0):.4f}"
    )

    print(
        f"Recall    : "
        f"{recall_score(y_test, test_pred, zero_division=0):.4f}"
    )

    print(
        f"F1        : "
        f"{f1_score(y_test, test_pred, zero_division=0):.4f}"
    )

    print(
        f"ROC-AUC   : "
        f"{roc_auc_score(y_test, test_scores):.4f}"
    )

    print("\nConfusion matrix:")

    print(
        confusion_matrix(
            y_test,
            test_pred
        )
    )

    print(
        f"\nFinal prediction time: "
        f"{prediction_time:.4f} seconds"
    )

    return {
        "model": nn,
        "threshold": threshold,
        "val_scores": val_scores,
        "test_scores": test_scores,
        "val_predictions": val_pred,
        "test_predictions": test_pred,
        "training_time": training_time,
        "prediction_time": prediction_time
    }


# ================================================================
# 19. RUN FINAL EVALUATIONS
# ================================================================

bcga_svm_eval = evaluate_bcga_svm(
    selected_features=bcga_svm_result["selected_features"],
    X_train=X_train_all,
    y_train=y_train,
    X_val=X_val_all,
    y_val=y_val,
    X_test=X_test_all,
    y_test=y_test
)


bcga_knn_eval = evaluate_bcga_knn(
    selected_features=bcga_knn_result["selected_features"],
    X_train=X_train_all,
    y_train=y_train,
    X_val=X_val_all,
    y_val=y_val,
    X_test=X_test_all,
    y_test=y_test
)


# ================================================================
# 20. FEATURE REDUCTION SUMMARY
# ================================================================

print("\n\n")
print("=" * 70)
print("PHASE E — FEATURE REDUCTION")
print("=" * 70)

svm_selected = len(
    bcga_svm_result["selected_features"]
)

knn_selected = len(
    bcga_knn_result["selected_features"]
)

svm_reduction = (
    1 -
    svm_selected /
    len(FEATURE_COLUMNS)
) * 100

knn_reduction = (
    1 -
    knn_selected /
    len(FEATURE_COLUMNS)
) * 100


print("\nONE-CLASS SVM")

print(
    f"  {len(FEATURE_COLUMNS)} → "
    f"{svm_selected} features"
)

print(
    f"  Reduction: "
    f"{svm_reduction:.1f}%"
)


print("\nKNN")

print(
    f"  {len(FEATURE_COLUMNS)} → "
    f"{knn_selected} features"
)

print(
    f"  Reduction: "
    f"{knn_reduction:.1f}%"
)


# ================================================================
# 21. FINAL SUMMARY TABLE
# ================================================================

print("\n\n")
print("=" * 70)
print("PHASE E — BCGA SUMMARY")
print("=" * 70)

def get_metrics(y_true, predictions, scores):

    return {
        "Accuracy": accuracy_score(
            y_true,
            predictions
        ),

        "Precision": precision_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "Recall": recall_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "F1": f1_score(
            y_true,
            predictions,
            zero_division=0
        ),

        "ROC-AUC": roc_auc_score(
            y_true,
            scores
        )
    }


svm_metrics = get_metrics(
    y_test,
    bcga_svm_eval["test_predictions"],
    bcga_svm_eval["test_scores"]
)

knn_metrics = get_metrics(
    y_test,
    bcga_knn_eval["test_predictions"],
    bcga_knn_eval["test_scores"]
)


summary = pd.DataFrame({

    "Metric": [
        "Original features",
        "Selected features",
        "Reduction (%)",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC-AUC"
    ],

    "SVM BCGA": [
        len(FEATURE_COLUMNS),
        svm_selected,
        svm_reduction,
        svm_metrics["Accuracy"],
        svm_metrics["Precision"],
        svm_metrics["Recall"],
        svm_metrics["F1"],
        svm_metrics["ROC-AUC"]
    ],

    "KNN BCGA": [
        len(FEATURE_COLUMNS),
        knn_selected,
        knn_reduction,
        knn_metrics["Accuracy"],
        knn_metrics["Precision"],
        knn_metrics["Recall"],
        knn_metrics["F1"],
        knn_metrics["ROC-AUC"]
    ]
})


print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)


# ================================================================
# PHASE E.1 COMPLETE
# ================================================================

print("\n")
print("=" * 70)
print("PHASE E.1 COMPLETE")
print("=" * 70)

print("""
Binary Chaotic Genetic Algorithm selected feature subsets
have been evaluated on the untouched final test set.

Next phase:
  Phase F — Comparative Analysis
""")

PHASE E — BINARY CHAOTIC GENETIC ALGORITHM

Candidate features:
   0: id.resp_p
   1: orig_pkts
   2: orig_ip_bytes
   3: resp_pkts
   4: resp_ip_bytes
   5: duration
   6: orig_bytes
   7: resp_bytes
   8: missed_bytes

Total candidate features: 9


ValueError: Missing features from train_df: ['orig_ip_bytes', 'resp_ip_bytes', 'missed_bytes']